# 📚 Letrova — Módulo 1: Clasificación Inteligente de Libros
### Nicaury Díaz · 23-SISN-2-028


## Celda 1 — Instalación de dependencias

In [9]:
!pip install transformers datasets torch scikit-learn -q
!pip install sentence-transformers faiss-cpu -q
!pip install chromadb rapidfuzz -q

import torch
print('=' * 55)
print('GPU disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Dispositivo    :', torch.cuda.get_device_name(0))
else:
    print('Dispositivo    : CPU')
print('=' * 55)

import transformers, sentence_transformers, faiss, chromadb
print('transformers         :', transformers.__version__)
print('sentence-transformers:', sentence_transformers.__version__)
print('faiss                : OK')
print('chromadb             :', chromadb.__version__)
print('\n Todas las librerías instaladas correctamente')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 79.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60

## Celda 2 — Imports, constantes y clases base

In [10]:
import requests, re, time, random, os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer
import faiss
import chromadb

GENEROS = {0:'Novela', 1:'Cuento', 2:'Poesía', 3:'Ensayo',
           4:'Teatro', 5:'Fábula', 6:'Crónica'}
NUM_LABELS = 7

TIPOS_LECTURA = {0:'Infantil', 1:'Juvenil', 2:'Académica', 3:'Entretenimiento'}
NUM_TIPOS = 4

MODEL_NAME = 'bert-base-multilingual-cased'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class DatasetLiterario(Dataset):
    def __init__(self, textos, etiquetas, tokenizer, max_length=128):
        self.textos, self.etiquetas = textos, etiquetas
        self.tokenizer, self.max_length = tokenizer, max_length
    def __len__(self): return len(self.textos)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.textos[idx], max_length=self.max_length,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        return {'input_ids': enc['input_ids'].squeeze(),
                'attention_mask': enc['attention_mask'].squeeze(),
                'label': torch.tensor(self.etiquetas[idx], dtype=torch.long)}

# LOOP DE ENTRENAMIENTO
def loop_entrenamiento(modelo, ld_train, ld_val, nombre_modelo, epochs=5):
    criterio  = torch.nn.CrossEntropyLoss()
    optimizer = AdamW(modelo.parameters(), lr=2e-5, weight_decay=0.01)
    total     = len(ld_train) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total*0.1), num_training_steps=total)
    mejor = 0
    print(f'Pasos: {total}  |  Warmup: {int(total*0.1)}')
    print('Iniciando entrenamiento...\n' + '='*55)
    for epoch in range(epochs):
        # Train
        modelo.train()
        p_t, c_t, n_t = 0, 0, 0
        for b in ld_train:
            ids, mask, lbl = b['input_ids'].to(device), b['attention_mask'].to(device), b['label'].to(device)
            optimizer.zero_grad()
            out = modelo(input_ids=ids, attention_mask=mask)
            loss = criterio(out.logits, lbl)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            p_t += loss.item(); c_t += (out.logits.argmax(1)==lbl).sum().item(); n_t += len(lbl)
        # Val
        modelo.eval()
        p_v, c_v, n_v = 0, 0, 0
        with torch.no_grad():
            for b in ld_val:
                ids, mask, lbl = b['input_ids'].to(device), b['attention_mask'].to(device), b['label'].to(device)
                out = modelo(input_ids=ids, attention_mask=mask)
                loss = criterio(out.logits, lbl)
                p_v += loss.item(); c_v += (out.logits.argmax(1)==lbl).sum().item(); n_v += len(lbl)
        acc_v = c_v/n_v
        print(f'Época {epoch+1}/{epochs}  Train: {c_t/n_t*100:.1f}%  Val: {acc_v*100:.1f}%')
        if acc_v > mejor:
            mejor = acc_v
            torch.save(modelo.state_dict(), nombre_modelo)
            print(f'   Guardado — {mejor*100:.1f}%')
    print(f'\n Entrenamiento completo. Mejor: {mejor*100:.1f}%')
    return mejor

print('Setup completo  |  Dispositivo:', device)

Setup completo  |  Dispositivo: cuda


## Celda 3 — Datos de entrenamiento (Gutenberg + ejemplos manuales)

In [ ]:
def descargar_fragmentos(libro_id, num_fragmentos=30, chars=300):
    try:
        r = requests.get(
            f'https://www.gutenberg.org/cache/epub/{libro_id}/pg{libro_id}.txt',
            timeout=15)
        texto = r.text[3000:]
        paso  = max(1, len(texto) // num_fragmentos)
        return [texto[i*paso:i*paso+chars].strip()
                for i in range(num_fragmentos) if len(texto[i*paso:i*paso+chars].strip()) > 80]
    except: return []

libros = {
    0: [1342, 84, 1661, 11, 2701, 98, 1400, 174, 5200, 4300, 768, 600, 2591, 1952, 730, 35, 1184],
    1: [2147, 514, 219, 345, 1064, 2148, 135, 3825, 1260, 23, 209, 2852, 158, 1527, 512],
    2: [1065, 1236, 23684, 19, 1020, 1041, 1322, 1934, 2264, 932, 16328, 1243, 3090, 18581, 8800, 18865, 1000],
    3: [16, 1232, 2130, 574, 3207, 1404, 2003, 844, 1321, 2009, 6409, 4200, 1690, 2010, 132, 147, 205],
    4: [1524, 1533, 1513, 1514, 27761, 1532, 1515, 1516, 2235, 1519, 1522, 1521, 1534, 1525, 1531, 2386, 1112],
    5: [11339, 19537, 7102, 28498, 28516, 1083, 2778, 7596, 7597, 17208, 8974, 18800, 18806, 18807, 18808, 28020, 28519],
    6: [23700, 2383, 128, 7243, 1430, 26740, 35997, 2781, 9200, 2048, 3296, 6843, 23685, 26268, 5765, 9462, 30254],
}
datos_textos, datos_etiquetas = [], []
print('Descargando fragmentos de Gutenberg...')
for etiqueta, ids in libros.items():
    frags = []
    for lid in ids: frags.extend(descargar_fragmentos(lid, num_fragmentos=35))
    random.shuffle(frags); frags = frags[:520]
    datos_textos.extend(frags); datos_etiquetas.extend([etiqueta]*len(frags))
    print(f'  {GENEROS[etiqueta]}: {len(frags)} fragmentos')

# EJEMPLOS MANUALES GÉNERO
ejemplos_genero = {
    0: [  # Novel
        'Elena had spent twenty years building the life that others expected of her. Every morning she woke up in the same bed, stared at the same peeling ceiling, and thought that someday everything would change. It wasn\'t until Tuesday afternoon, when she found the folded letter inside the book her mother had left her when she died, that she understood change would not come from outside but from a decision she had been postponing for years. She read the words three times and cried for the first time in months.',
        'Captain Rodrigo Salvatierra had sailed for forty years without losing a single cargo, but that night off the coast of Cádiz he felt fear for the first time in his life. The storm was not on any map. His men looked at him from the deck, waiting for an order he had not yet been able to formulate. He thought of his daughter, of the promise he had made to return before Christmas, and gripped the helm with both hands, knowing that this could be his last decision as captain.',
        'The first time Mateo saw Clara was on the platform of the central station, a Thursday in November when the world seemed to have turned completely gray. She was reading a book with the cover torn off, and he wondered what kind of person goes through life without knowing the title of what they read. Four years later, sitting on the same station with his luggage at his feet, he still could not answer that question. Nor could he explain how someone so impossible had become the center of all his plans.',
        'During the three days that followed her father\'s death, Inés did not say a single word. Her sisters watched her from the doorway of the room, not daring to enter, not knowing what to say to someone who had spent half her life in silence with that man and the other half running away from him. The notary arrived on Thursday. The papers were clear: the house, the debt, the abandoned estate in the north. All for Inés. As if her father had known, even at the end, that she was the only one who could bear the weight of that story.',
        'Marco Aurelio Bernal arrived in the town with two suitcases and the conviction that no one would recognize him. He had spent thirty years in the capital remaking his name, his accent, even the way he walked. But when the woman at the grocery store looked at him fixedly and said without preamble, "You are the youngest son of the Bernal family, the one who left," he understood that some things are not erased by time or willpower. He paid for the bread without answering and walked out into the street knowing that in that town every stone held a memory he preferred not to awaken.',
        'The novel that Sofía had been writing for five years had four hundred pages and three different endings, none of which satisfied her. Her editor called every two months with the patience of someone who has learned to wait. Her ex‑husband had once told her that the problem was not the book but that she didn\'t want to finish anything. Sofía had stored that sentence in a mental drawer for years. The afternoon she finally wrote "THE END" in capital letters with a period, she understood that he had been right, but also that it was time to prove him wrong.',
        'The summer I turned twelve was the last summer of my childhood, though I didn\'t know it then. My grandmother died in July, my father lost his job in August, and by September we had sold the beach house that had been in the family for four generations. I remember the last day there, the way the sun came through the wooden blinds and drew stripes of light on the white ceramic floor. I stood in the doorway for a long while, thinking I should etch every detail into my memory, as if I already knew that places like that only exist once.',
        'Victoria Salomé Arenas took exactly sixteen years to gather enough courage to return to the neighborhood where she had grown up. It was not nostalgia that brought her back but an envelope with her name handwritten on it, found in the mailbox of her apartment, no return address, postmarked from a city she believed held nothing for her anymore. Inside was a photograph and three words. That was enough for her to unpack the suitcase she had had half‑packed for three weeks and buy a bus ticket for the next day.',
        'Detective Aurelio Ríos had a reputation as the best in the city and the worst person to meet at a dinner party. His colleagues said he could spot a lie in twelve seconds but was incapable of holding a trivial conversation for more than two minutes. What no one knew was that behind that calculated coldness there was a man who kept the same photograph in the inside pocket of his coat for twenty‑two years: a light‑eyed girl who had at some point stopped being an unsolved case and become the reason he still did this job.',
        'Forty years had passed since the three Villanueva brothers had last been under the same roof. The occasion had been Ramón’s wedding, the eldest. Now it was his funeral that brought them back together in the old family house, with their respective silences, their respective grievances, and the children they had had with people the others had never fully accepted. The cook, who had been in the house for thirty years, watched the scene from the kitchen door and thought that some families only know how to be together when they have no other choice.',
        'When Professor Adela Montoya discovered that one of her fifth‑grade students had been writing a novel during recess, her first instinct was to ask him to put away his notebook and pay attention to class. But something stopped her. She read the first few pages standing by the window while the other children played outside, and she knew she was holding something that cannot be taught or learned but simply appears in certain people without warning and without asking permission.',
        'The inheritance left by Grandfather Fermín was neither money nor land but a wooden box with a brass lock that no one in the family had been able to open for forty years. When they finally opened it on the anniversary of his death, they found no treasure inside but fifty‑three love letters written in a language that none of his descendants recognized. It took them two weeks to discover they were written in Catalan and another six months to understand what their grandfather had been doing during the ten years he claimed to be away on business.',
        'Santiago had been sleeping on his best friend’s couch for three weeks when he decided it was time to do something with his life. He didn’t know exactly what, but he was thirty‑four years old, with a university degree no one had ever asked him to use, and the unsettling feeling that the world had kept turning while he was busy waiting for something to change on its own. That morning he showered, put on the shirt with the fewest wrinkles, and walked out the door with no concrete plan, which was already a significant change from the previous days.',
        'The manuscript arrived at the publishing house on a Monday in October with no author’s name, no return address, and only one instruction written on the first page: "Read it before you reject it." The literary director read it that same afternoon intending to return it the next day. By three in the morning, unable to stop, she understood that the instruction was not arrogance but a completely justified warning, and that she would have to explain to her team why they were going to reorganize the next year’s catalog.',
        'Luisa Fernanda had promised herself she would not fall in love with anyone who lived more than fifty kilometers away. It was a reasonable promise, born of experience and the weariness of spending years loving people who inhabited other time zones. The problem was that Carlos lived twelve hundred kilometers away and that their first conversation lasted four hours without either of them noticing how time had passed. By the time she remembered the promise, it was already too late and, besides, too unimportant.',
        'The last summer before everything changed, the four friends made a pact: they would meet again in that exact place ten years later, no matter what had happened in their lives. None of them knew then how much a person can change in a decade, nor how difficult it is to keep promises made when you still believe that distance does not affect the people you love. Ten years later, three of the four arrived promptly at the agreed place. The fourth sent a letter.',
        'The first night Esperanza spent in the convent she did not sleep. It was not fear or regret but the strange sensation that silence had its own texture, almost tangible, completely different from the silence she had known in the city. In the city, silence was always provisional, interrupted by something before you could get used to it. Here silence was the norm and sound the exception, and that fundamentally changed one’s relationship with oneself.',
        'The doctor told him he had six months, and Ernesto spent the first three deciding how he was going to use the last ones. It was not a dramatic decision but a surprisingly practical one: he made a list, crossed off the things that were not really important, and ended up with seven. The first was to call his brother, whom he had not spoken to in twelve years. He did it that same afternoon. The conversation lasted forty minutes, and when it ended neither of them knew exactly why they had waited so long to have it.',
        'Catalina Espinosa arrived in Madrid with four hundred euros, a Spanish learned from movies, and the certainty that her life began there. What no one had explained to her was that starting from zero in a city that doesn’t know you is exactly that: zero. No connections, no references, no understanding of how everything works. Just a rented room, a job in a café, and the slow realization that courage guarantees nothing but simply gives you the opportunity to find out.',
        'The writer who never published anything died leaving fourteen complete novels in a trunk under his bed. His niece, who inherited the apartment, took three days to open the trunk and another two weeks to read the first of the manuscripts. When she finished, she could not understand why that man had chosen to keep something so extraordinary to himself. Then she found the answer in the last notebook: a sentence written in small letters that said simply, "Some things are too personal to share."',
    ],
    1: [  # Short story
        'The cobbler opened his shop on Monday and found that someone had repaired all the shoes he had left on the counter overnight. The same thing happened on Tuesday and Wednesday. On Thursday he decided to stay awake to discover who it was. He fell asleep at two in the morning. On Friday the shoes were perfect again. He stopped trying to find out and instead left a plate of food next to the counter every night. It was always gone by morning.',
        'A woman received the same telegram twice: once on her wedding day and again exactly thirty years later. The text was identical. The sender was the same. The only problem was that person had died twenty‑five years earlier, and the telegram was postmarked that very morning.',
        'The man who never dreamed woke up one Tuesday with the absolute certainty that he had dreamed something important. He did not remember the dream but knew it was important. He spent the day waiting for the memory to return. It did not. The next night he dreamed the same thing and forgot it again. This went on for three weeks until one morning he woke up crying and for the first time remembered everything. It was simply his mother’s face.',
        'The inventor built a machine that could answer any question. The first question they asked it was: "Does God exist?" The machine took ten seconds, which was much longer than usual. Then it replied: "Now it does." The inventor understood too late what he had done.',
        'The village library had a book that no one could finish. Not because it was boring, but because the last pages changed every time someone read them. The librarian had spent forty years trying to catalog it. On its card she simply wrote: "Author unknown. Genre: undetermined. Pages: variable."',
        'A man spent twenty years searching for the best restaurant in the world. He visited 340 countries, ate in 1,200 Michelin‑starred restaurants, and wrote a book about it. In the epilogue he confessed that the best meal he had ever tasted was a chicken and rice dish his mother had made in 1987 in an aluminum pot on a gas stove in a windowless apartment.',
        'The little girl asked her grandfather why he kept all the empty shoeboxes under his bed. Her grandfather told her that inside were all his memories. The little girl looked at the closed boxes and asked if she could open them. Her grandfather said no, that those memories were too old to survive the air.',
        'The musician played the same tango every night for thirty years in the same café. No one ever applauded. No one ever commented. The last night, before closing the café forever, he played the tango one last time. When he finished, the owner of the café, who had never said a word to him, simply said, "That was the best tango I have ever heard in my life." Then he turned off the lights.',
        'The tree planted by the great‑grandfather grew until it touched the window of his granddaughter’s room. One night the branch tapped the glass three times, as if knocking at the door. The granddaughter opened the window and among the leaves found a letter written in handwriting she recognized as her great‑grandfather’s, even though he had died thirty years before she was born.',
        'There was a tradition in the village: on the first day of the year, each inhabitant had to burn a piece of paper on which they had written their greatest fear. For generations they did it without question. One year, a ten‑year‑old boy asked where the burned fears went. No one could answer. That night several adults in the village could not sleep.',
        'The painter discovered that all her paintings had an error: in each one there was a tiny figure that she did not remember painting. She checked all the paintings she had sold over the past ten years. The same figure was in all of them. Always in the same place. Always looking out of the painting.',
        'The watchmaker who repaired broken watches developed over time the ability to know, by listening to the tick‑tock of a watch, how much time its owner had left. He never told anyone what he heard. When he finally retired and closed his shop, he discovered that his own watch had been silent for years.',
        'Two strangers met on the same park bench for five consecutive years, always on the same day, always at the same time, knowing nothing about each other. The first year they ignored each other. The second they nodded. The third they commented on the weather. The fourth they talked for two hours. The fifth neither went to the park, but both thought about the other all day long.',
        'The cook prepared each dish thinking of a different person. He did not do it consciously, but those who ate at his restaurant inevitably left thinking of someone they had not called in a long time. In twenty years of work he never understood why his food had that effect. Neither did his customers.',
        'A man inherited from his father a compass that pointed south. He lived his whole life in the north thinking the compass was broken. At eighty, for the first time, he followed the direction it marked. After three days of walking he arrived at the village where his father was born, where no one knew him but everyone recognized him.',
        'The translator discovered that the book she had been translating for three years did not exist in its original language. She checked the archives, consulted colleagues, searched catalogs around the world. There was no trace of the original. Only her translation, which every time she read it seemed more familiar, as if she had written it herself at another time in her life.',
        'The astronaut returned from six months in space and the first thing he did was call his mother. They talked for an hour about completely ordinary things: the garden, the neighbor, the cat that had died while he was away. When he hung up, the astronaut thought that phone call had been the most extraordinary experience of the whole journey.',
        'A hotel had a room that no one could book because it always appeared occupied in the system. The staff paid no attention. A curious new receptionist forced the lock. The room was perfectly tidy, with the bed made and a glass of water on the nightstand. The water was still cold.',
        'The woman who collected goodbyes spent forty years going to airports, bus terminals, and train stations to watch people part. She never traveled. When asked why, she said that airports are the only places in the world where people say exactly what they feel.',
        'The postman who delivered mail in the same neighborhood for thirty years knew before the recipients whether a letter brought good or bad news. Not by magic, but because in thirty years he had learned to read the weight of the paper, the pressure of the pen, and the particular smell that letters written with fear have.',
    ],
    2: [  # Poetry
        'Time aches like bones ache in winter,\nlike light aches when you have stayed too long in darkness.\nI look for you in corners where you are no longer,\nin the coffee that grows cold on the table,\nin the silence you left where your voice used to be.',
        'I want to be the bread you bake each morning,\nthe steam rising from fresh coffee,\nthe scent of earth after the October rain.\nI want to be the ordinary things\nyou touch without thinking\nbecause they are part of the air you breathe.',
        'Death is not an abyss but a door\nthat closes from the inside\nand no one from outside can ever open.\nEveryone I loved has crossed that threshold\nand I stay here in the garden\nlearning to live with closed doors.',
        'Some cities enter through the eyes\nand others through the skin.\nThis city entered through my hands,\nthrough the touch of cobblestones in the rain,\nthrough the texture of stone walls\nthat hold centuries of stories no one told me.',
        'The poem I did not write\nis the longest of all,\nthe one that occupies the space between your words,\nthe one that exists in the second before speaking\nwhen everything is still possible\nand language has not yet spoiled anything.',
        'Autumn returns with its liturgy of leaves,\nwith its promise of fire and farewell,\nwith that particular scent of something ending\nand something else that has not yet begun.\nAutumn returns and I become again\nthe child who watched leaves fall thinking\nthe world had something to tell me.',
        'I write to you from the country of insomnia,\nwhere hours are longer\nand thoughts more honest.\nHere at three in the morning\neverything I avoided during the day\nsits beside me again\nwith the patience of one who knows I cannot leave.',
        'My grandmother spoke to plants\nthe way one speaks to people who listen well:\ncalmly, without hurry,\nknowing they will not interrupt.\nI learned from her that silence\nneeds to be cultivated,\nthat growing toward the light is also a form of love.',
        'Yesterday I found your name written\nin the margin of a book we read together.\nYour handwriting was smaller then,\ntighter, as if you wanted\nto fit into the smallest possible space.\nToday your handwriting has opened up.\nToday you no longer fit in any margin.',
        'Rain does not know where it is going:\nit falls and that is enough for it.\nI, on the other hand, go through life\nlooking for destinations, reasons, meanings,\nwhile rain soaks me without asking\nif I give it permission.',
        'Every language I learned\nwas a new room in the house that I am.\nIn one I keep my dreams,\nin another the words for hunger,\nin a third the phrases of love\nthat in my native tongue\nalways sounded too big for me.',
        'There is an exact distance\nfrom which things\nare seen in all their beauty\nwithout hurting too much.\nI have spent years looking for that distance\nand when I find it\nsomething always brings me closer again\ntoo close.',
        'The sea has no memory\nbut each wave brings\nsomething of all the waves before.\nThat is how I am with you:\nI do not remember every moment we lived\nbut every time I see you\nI carry all the previous water inside me.',
        'Talking to you is like opening a window\nin a room that had been closed too long.\nThe air I did not know I needed enters,\nthe light enters and makes visible\nthe specks of dust floating,\nthe world that was out there waiting\nfor someone to have the courage to open.',
        'I am the daughter of those who crossed the sea\nwith one suitcase and a name memorized.\nI am the daughter of the one who arrived without a language\nand of the one who learned to be silent in two tongues.\nI am the accent that never quite leaves\nnor quite stays,\nthe bridge between two shores\nneither of which claims me completely.',
        'When I was little I believed\nthat adults knew where they were going.\nNow that I am an adult I know\nthat no one goes anywhere with certainty,\nthat we all move forward groping in the dark,\nlooking for the wall with my hand outstretched,\ngrateful when we find something solid to hold on to.',
        'Photographs do not freeze time:\nthey show it dying.\nThat moment stopped on paper\nis proof that it no longer exists,\nthat the child in the photo\nhas gone on becoming an adult\nwhile the image remained that same child.',
        'Forgive me for writing to you without knowing\nif what I feel has a name\nor is simply that strange weight\nthat settles in the chest on some Sundays\nwhen the day closes too early\nand the light leaves without warning.',
        'I have learned that some people\ncome into your life like winter light:\noblique, cold to the touch,\nbut without it the room would be impossible.\nYou are that light.\nNot the warmth I was looking for:\nthe kind of light without which\nI could not see anything.',
        'The tree we planted together that April\nis now taller than us.\nIt grew without either of us\nstaying to watch it grow.\nI suppose that is also a form of kinship:\ntaking care of something knowing\nthat you will not see the full result of your care.',
    ],
    3: [  # Essay
        'The central problem of contemporary education lies not in the amount of content transmitted but in the absence of instruments to assess what really matters: the ability to think rigorously, tolerate uncertainty, and formulate questions before answers. As long as educational systems continue to measure data retention as a proxy for intelligence, we will keep training students who know many things but do not know what to do with what they know.',
        'There is a paradox at the heart of liberal democracy that is rarely examined with the seriousness it deserves: the system that proclaims the political equality of all citizens operates in practice through mechanisms that concentrate decision‑making power in technical and financial minorities. This tension is not accidental but structural, and no reform that does not confront it directly can be considered genuinely democratic.',
        'The question of whether artificial intelligence can be creative depends entirely on what we understand by creativity. If we define it as the capacity to produce novel results from learned patterns, then AI systems are creative in a technically precise sense. If, on the other hand, we define it as the capacity to interrupt one’s own patterns and start from a genuine breaking point, then creativity remains, for now, an exclusively human territory.',
        'Collective memory does not function as an archive but as a set of narratives in constant renegotiation. What a society remembers of its past is not a faithful copy of what happened but a selection conditioned by the needs of the present: dominant groups tend to privilege memories that legitimize their position, while dissident memories survive on the margins until political conditions allow their reappearance.',
        'There is a fundamental difference between knowing that something is true and having integrated that knowledge in a way that modifies behavior. The gap between declarative knowledge and lived knowledge is one of the most persistent problems in any process of human formation, from formal education to psychological therapy. Knowing that sugar is harmful does not change the habit of consuming it; knowing that exercise is beneficial does not automatically turn anyone into someone who exercises.',
        'The essay as a literary form is, in its best version, an act of public thinking: the writer thinks aloud and the reader accompanies them in that process, not to receive conclusions but to participate in the movement of intelligence. Unlike the academic article, which presents results, or the manifesto, which proclaims positions, the essay allows itself the intellectual luxury of not quite knowing where it is going as it writes, trusting that the path itself will reveal something that the starting point did not contain.',
        'The attention economy has radically transformed the relationship between the individual and information. In a context where thousands of messages compete simultaneously for available cognitive time, sustained attention has become a scarce good and, paradoxically, a form of resistance. Reading a long book, watching a film without interruptions, having a deep conversation without checking one’s phone: these acts, once considered ordinary, are today small insurrections against the logic of the fragment.',
        'Every language we speak gives us access to a different region of experience. It is not that speakers of different languages live in incomparable worlds, as the strong Sapir‑Whorf hypothesis claimed, but it is true that certain experiences are more easily articulated in some languages than others. The loss of a language is not only a cultural impoverishment but a contraction of the possibilities of collective thought.',
        'Western moral philosophy has been dominated for centuries by the question of what we should do. An alternative tradition, ranging from Aristotle to contemporary ethics of care, proposes that the more fundamental question is who we should be. This distinction is not merely academic: depending on which question is taken as a starting point, radically different answers are obtained about responsibility, community, and the nature of the good life.',
        'Platform capitalism has produced a new category of workers who are simultaneously employees and customers, producers and consumers, people and data. This ambiguity is not a side effect of the model but its organizing principle: dissolving the boundaries between spheres that were previously differentiated allows value to be extracted from dimensions of human existence that previously remained outside the economic circuit.',
        'There is a widespread tendency to confuse complexity with depth and clarity with superficiality. This confusion benefits those who write obscurely and harms those who strive to make difficult ideas accessible. Clarity is not a concession but an ethical requirement: it means that one has understood enough to be able to explain it, and that one respects the reader enough not to hide behind technical language.',
        'Literature is not an ornament of thought but one of its most powerful instruments. The novel, in particular, is a cognitive technology that allows us to explore subjective experience from the inside, with a granularity and complexity that no other medium can reproduce. When we read quality fiction, we do not escape reality: we learn to inhabit it with greater richness and greater compassion.',
        'The concept of merit has historically served to justify inequalities that have structural causes by presenting them as results of individual differences. To say that someone occupies a privileged position because of their own merit requires systematically ignoring all the factors that conditioned their starting point: the zip code where they were born, the educational level of their parents, the social capital they had access to, the color of their skin in a society that still discriminates on that basis.',
        'Mass tourism has produced a paradox that its own practitioners rarely recognize: in seeking the authenticity of different places and cultures, it destroys it. The presence of millions of visitors inevitably transforms the places visited, turning what was everyday life into spectacle, what was local into global, what was particular into generic. The tourist seeks what has not yet been touched by tourism and contributes by that very search to making it touched.',
        'Science is not a set of established truths but a method for producing provisional knowledge that can be revised and corrected. This characteristic, which some consider a weakness, is in fact its greatest epistemological strength. A system that cannot update itself when new evidence appears is not more solid but more rigid, and rigidity in the presence of complexity invariably leads to error.',
        'Voluntary solitude and imposed solitude are radically different experiences that share the same name. The former is a choice that produces clarity, the latter a condition that produces deterioration. Confusing them has led to public health policies that treat chronic solitude with recipes intended for those who have chosen it, with the predictable result of not helping those who truly suffer from it.',
        'Language not only describes reality but constitutes it. The words we have to name an experience determine in part how we can think about it, discuss it, and eventually transform it. That is why struggles over language are frequently struggles over power: whoever controls the terms in which a problem is defined controls, at least partially, the space of solutions considered possible.',
        'The speed with which information circulates on digital networks has produced a paradoxical effect on knowledge: more data available has not produced more understanding but more noise, more fragmentation, and, in certain cases, more active ignorance. Unrestricted access to information does not by itself solve the problem of functional illiteracy; in some ways it worsens it, by providing materials for building worldviews without the critical tools to evaluate them.',
        'Utopia is not a political project but a cognitive function: it allows us to imagine that things could be different from what they are, which is the first necessary step for any real social transformation. A society that has lost the capacity to imagine alternatives to its own functioning is not being realistic but has internalized a form of voluntary servitude, confusing what exists with what is possible.',
        'Art does not have a social function in the sense that medicine or engineering does, and that is precisely what makes it necessary. A society that values only what has immediate and measurable utility amputates from itself an entire dimension of experience: the dimension of what cannot be reduced to an instrument, of what exists in its own right, of what reminds us that there are ways of being human that no algorithm can anticipate or replace.',
    ],
    4: [  # Theatre
        'ELENA: (long pause, looking out the window) It’s not that I don’t love you. It’s that I no longer know who I am when I’m with you.\nJULIÁN: (without moving) That’s the same as saying you don’t love me.\nELENA: No. It’s the opposite. With you I lose myself in a way that I like, and that scares me.\nJULIÁN: (finally looking up) Then stay.\nELENA: (long silence) I can’t stay precisely because I want to.',
        'COMMISSIONER: Tell me where you were on Tuesday night.\nSUSPECT: (without hesitation) At home.\nCOMMISSIONER: Alone?\nSUSPECT: I’m always alone.\nCOMMISSIONER: (taking notes) And no one can confirm it?\nSUSPECT: (looking fixedly at him) People who live alone don’t have alibis, Commissioner. That doesn’t make us guilty. It just makes us inconvenient for your job.',
        'MOTHER: When you left, you left the light in your room on.\nDAUGHTER: (without looking at her) I know.\nMOTHER: Why?\nDAUGHTER: (pause) Because I didn’t want it to look like I had left completely.\nMOTHER: (voice breaking) It’s been three years.\nDAUGHTER: (finally turning) I’ll turn it off when I come back.',
        'DOCTOR: (reviewing the file without looking at the patient) The results are not what we expected.\nPATIENT: (after a silence) How long?\nDOCTOR: (looking up for the first time) That depends on many factors—\nPATIENT: (interrupting) How long, Doctor.\nDOCTOR: (long pause) Six months. Maybe eight if you respond well to treatment.\nPATIENT: (nodding slowly) Good. That gives me enough time.',
        'JOURNALIST: Do you deny having met Mr. Marcos before the contract was signed?\nMINISTER: (with a smile) What I deny is the premise of your question.\nJOURNALIST: Could you be more specific?\nMINISTER: (the smile disappears) What I can be is more brief: I do not answer questions that assume unproven facts.\nJOURNALIST: So do you deny it or not?\nMINISTER: (standing up) This interview is over.',
        '(Scene: a waiting room. Two people sitting without looking at each other. Time passes. The light changes subtly.)\nFIRST WOMAN: Have you been waiting long?\nSECOND WOMAN: (without moving) Long enough to have decided several times to leave and stay.\nFIRST WOMAN: And what made you stay?\nSECOND WOMAN: (looking at her for the first time) I suppose the same thing that made you.',
        'GRANDFATHER: (to the boy) Do you know why I brought you here?\nBOY: (looking at the sea) Because you used to bring Dad here when he was little.\nGRANDFATHER: (surprised) He told you that?\nBOY: No. But I found it in a photo.\nGRANDFATHER: (long silence, looking at the water) Sometimes children know us better than we know them.\nBOY: (after a moment) Can I ask you something you might not want to answer?\nGRANDFATHER: (without hesitation) Yes.',
        'DIRECTOR: The budget doesn’t cover both projects.\nARCHITECT: Then we have to choose.\nDIRECTOR: No. We have to get more budget.\nARCHITECT: (pause) From where?\nDIRECTOR: That’s what’s called a creative problem.\nARCHITECT: (after a silence) It used to be called a problem.\nDIRECTOR: Before there was no solution. Now there is. We just haven’t found it yet.',
        'NEIGHBOR 1: They’ve been making that noise for three days.\nNEIGHBOR 2: I already called the building manager.\nNEIGHBOR 1: And?\nNEIGHBOR 2: He said he’d look into it.\nNEIGHBOR 1: That’s it?\nNEIGHBOR 2: Then he asked me if I was the one who played music on Fridays.\nNEIGHBOR 1: (pause) And do you?\nNEIGHBOR 2: (after a long pause) That is irrelevant to the case at hand.',
        'TEACHER: (to the group) Does anyone know why it’s called that?\n(Long silence.)\nTEACHER: No one? (Waiting.) Well. That means we are all going to learn something today.\nGIRL IN THE BACK: (without raising her hand) Because if we already knew the name we wouldn’t need the class.\n(The teacher looks at her for a long moment.)\nTEACHER: Exactly. What’s your name?',
        'FIRST VOICE: When was the last time you did something for the first time?\nSECOND VOICE: (thinking) Yesterday.\nFIRST VOICE: What did you do?\nSECOND VOICE: I apologized to someone I owed an apology to for ten years.\nFIRST VOICE: And how was it?\nSECOND VOICE: Like doing something for the first time. Uncomfortable. Necessary. And then, suddenly, lighter.',
        '(The room is in semi‑darkness. A woman with her back turned. A man enters.)\nMAN: You’ve been here for hours.\nWOMAN: (without turning) I know.\nMAN: What are you looking at?\nWOMAN: The street.\nMAN: Is something happening on the street?\nWOMAN: No. That’s why I’m looking at it. Because nothing is happening, and yet in here too much is happening.',
        'JUDGE: You have the right not to testify.\nACCUSED: I know.\nJUDGE: Do you wish to exercise that right?\nACCUSED: No.\nJUDGE: Are you sure?\nACCUSED: (looking at the jury) Completely. Because what I have to say is the truth, and the truth does not serve me to keep silent even if it condemns me.',
        'SOFÍA: Did you miss me?\nMARCOS: (pause) Yes.\nSOFÍA: Just yes?\nMARCOS: What more do you want me to say?\nSOFÍA: Something that doesn’t sound like an exam answer.\nMARCOS: (long pause, then slowly) I missed you the way you miss air when a room has been closed too long.\nSOFÍA: (after a moment) That’s better.',
        '(Hospital room. Cold light. A woman holds the hand of an unconscious man.)\nWOMAN: (in a low voice) You can go now. I’ve finished telling you everything I needed to tell you.\n(Long pause. The heart monitor makes its steady noise.)\nWOMAN: (even lower) Or almost everything.\n(Pause.)\nWOMAN: That money in the north bank account was always for you. I’m not keeping it.',
        'CANDIDATE: My proposal is simple: honesty.\nJOURNALIST: Can you be more specific?\nCANDIDATE: Not lying. Not promising what I can’t deliver. Not pretending I have answers I don’t have.\nJOURNALIST: And do you think that’s what the electorate wants?\nCANDIDATE: (pause) No. I think it’s what they need. Which is not the same thing.',
        'GRANDSON: Grandpa, what’s the hardest thing you’ve ever done in your life?\nGRANDFATHER: (without hesitation) Forgive.\nGRANDSON: Who?\nGRANDFATHER: (long pause) Myself.\nGRANDSON: Why?\nGRANDFATHER: For the things I did. And for the things I didn’t do when I should have done them. The latter weigh more.',
        '(Two sisters. An open suitcase on the bed.)\nOLDER SISTER: You don’t have to go.\nYOUNGER SISTER: (folding clothes without looking at her) Yes, I do.\nOLDER SISTER: Why?\nYOUNGER SISTER: (stops, finally looks at her) Because if I don’t go now, in ten years I’ll still be wondering what would have happened if I had gone.\nOLDER SISTER: (after a silence) Call when you get there.',
        'CUSTOMER: I want to return this book.\nBOOKSELLER: (looking at the book) It’s already read.\nCUSTOMER: Yes.\nBOOKSELLER: We don’t accept returns on read books.\nCUSTOMER: I know. But it changed my life, and I’m not sure I want this kind of life.\nBOOKSELLER: (pause, then taking the book) What did it change?\nCUSTOMER: Everything. That’s why I need to return it.\nBOOKSELLER: (opening the book) Tell me.',
        '(End of play. The stage is almost dark. Only a voice.)\nVOICE: Everything we did today existed. No performance is repeated exactly the same. This performance, this moment, these words in this order in front of these people: never again. (Pause.) That is what makes theater different from all the other arts. That it disappears. That it leaves only what remains in those who experienced it.',
    ],
    5: [  # Fable
        'A thirsty crow found a jar with very little water at the bottom, too low to reach with its beak. It thought for a long time and then began dropping pebbles into the jar, one by one, patiently. The water slowly rose until it could drink. The animals watching laughed at first and fell silent at the end. Moral: Intelligence achieves what strength cannot, and patience is the first requirement of any true solution.',
        'The hare told the tortoise that a race between them was a joke. The tortoise accepted without argument. When the hare, confident in its advantage, decided to rest halfway, the tortoise continued moving at its own pace without stopping once. When it awoke, the hare ran to the finish line and found the tortoise already sitting there waiting. Moral: Speed without constancy is worth less than slowness without pauses, and those who believe themselves superior are often the first to be surpassed.',
        'A frog living in a well believed the well was the whole world and told anyone who would listen. A swallow passing by on its way to the sea tried to explain that there were oceans larger than the well. The frog looked at it with pity and said that was impossible because nothing could be bigger than the place where one lived. Moral: Those who know only their own world have the certainty of those most in need of doubt.',
        'A dog crossing a bridge with a bone in its mouth saw its reflection in the river. Believing it was another dog with a larger bone, it dropped its own to take the one in the water. The bone sank and the reflection vanished. The dog went home with nothing. Moral: Whoever covets what another has risks losing what they already possess, and ambition without discernment is the surest path to loss.',
        'The wind and the sun argued about which was more powerful. To settle it, they agreed on a contest: whoever could make the traveler walking down the road take off his coat would win. The wind blew with all its might but the traveler wrapped himself tighter. The sun began to shine gently and the traveler, feeling warm, took off his coat willingly. Moral: Gentle persuasion achieves more than violent force, and warmth wins where force fails.',
        'The mice in the barn gathered to decide what to do about the cat that threatened them. After many ideas, the youngest mouse proposed tying a bell around the cat’s neck to hear it coming. Everyone applauded the solution. Then the oldest mouse asked: who among us will put the bell on the cat? No one answered. Moral: It is easy to propose brilliant solutions that others must execute, and harder to take responsibility for what one has proposed.',
        'A fox lost its tail in a trap and, ashamed, tried to convince all the foxes in the forest that tails were a useless burden that no animal of good judgment should carry. Some were about to be convinced when one of the oldest asked: would you give us that advice if you still had your tail? The fox had no answer. Moral: Distrust those who propose we give up something they no longer have.',
        'A farmer found an eagle’s egg fallen from its nest and placed it with his hens’ eggs. The eaglet grew up among chickens and learned to walk, eat, and behave like a chicken. One day it saw an eagle soaring in the sky and felt something strange in its chest. It asked what that majestic animal was. It was told it was an eagle, king of birds, but that it was a chicken and chickens do not fly. It died believing it was a chicken. Moral: What we believe we are determines what we are capable of doing.',
        'Two goats met on a narrow bridge, one coming from each side, and neither would give way. They pushed, threatened, and finally both fell into the river. The shepherds watching from the banks remarked that if either had yielded for a moment, both would have reached the other side without any trouble. Moral: Pride that does not yield exacts a price that neither party can pay alone.',
        'A stork invited a fox to dinner and served the food in a tall, narrow‑necked dish from which the stork could eat but the fox could not. The fox, remembering that he had done the same to the stork earlier by serving her in a flat dish, said nothing. He ate what he could and went home hungry. Moral: Those who do to others what they would not want done to them should not be surprised when they receive the same treatment.',
        'The king of the animals summoned everyone to divide up the forest territory. The lion, the bear, and the wolf arrived confidently. The rabbit arrived late. When the rabbit asked for its share, the lion told it that everything had already been distributed. The rabbit asked what rule established that arriving late meant having no share. The forest fell silent. There was no rule. Moral: Rules applied without having been explained usually exist to benefit those who apply them.',
        'A warhorse was sent to work in the fields after the battles ended. It constantly complained about its new life, remembering the glory of past times. An ox plowing beside it said: the glory you remember is already gone. The field we work today exists. The horse kept complaining for years. The ox kept plowing. Moral: Living in the past does not cultivate anything in the present, and nostalgia misused is a form of immobility.',
        'A crow envying the peacock’s plumage gathered fallen feathers and stuck them on to look like a peacock. While strutting proudly before the other animals, the feathers began to fall off one by one. The peacocks rejected it because it was not one of them. The crows did too because it had scorned them. It ended up alone between the two groups, belonging to neither. Moral: Those who reject their own nature to resemble another often lose both identities and gain none.',
        'The ant worked all summer storing food for winter while the grasshopper sang and rested. When winter came, the grasshopper came begging for food. The ant asked what it had done during the months of abundance. The grasshopper said it had made music to cheer the summer. The ant replied: then dance now to warm the winter. Moral: Present work is the only guarantee of future well‑being, and no one can collect what they have not earned.',
        'A young elephant believed it was the largest animal in the world because it had never left its meadow. One day it was taken to the sea and saw a whale for the first time. It was silent for a long time. When it spoke again, it never again said it was the biggest at anything. It only asked: and what lies beyond this? Moral: Knowledge is measured not by what one knows but by the understanding of how much remains to be known.',
        'A snake borrowed from its neighbors promising to repay with interest, but never paid. One day no one would lend it anything anymore. The snake said the world was unfair to it. The owl listening from its tree replied: the world is exactly as fair as the trust you have built, no more and no less. The snake left muttering that the owl understood nothing. Moral: Those who destroy trust also destroy the credit their future depends on.',
        'The monkey who judged two cats disputing a piece of cheese decided to cut the cheese to share it equally. But one half turned out larger. To make them equal, it bit a bit from that half. Then the other half became larger. So it bit from one side and the other until the cheese was gone. The two cats looked at it empty‑handed. Moral: Those who mediate in others’ disputes with self‑interest end up being the only beneficiaries, and sometimes not even that.',
        'A deer admired its antlers in the river’s reflection but cursed its slender, fast legs. One day a hunter chased it through the forest. The slender legs saved it. But in fleeing, the antlers got tangled in the branches and the hunter nearly caught it. It escaped only barely. Moral: What we despise in ourselves may be our salvation, and what we admire may be our ruin, because the value of each thing depends on the moment it is needed.',
        'A caged bird that sang beautifully was freed by its owner after many years. The first days it flew in small circles as if the cage were still there. Over time the circles grew larger. Finally it flew without limits. But it never flew as far as the birds that had always been free. Moral: Freedom that comes late shapes those who receive it differently than freedom that was always there, and there are things that lost time does not fully restore.',
        'A donkey loaded with salt crossed a river and discovered that when part of the load sank, it dissolved and the weight decreased. It began to throw itself into the river on purpose every time it crossed. Its owner noticed and loaded it with sponges. The next time the donkey threw itself, the sponges soaked up water and the weight tripled. The donkey could barely get out of the water. Moral: Those who repeat a strategy without asking whether the conditions are still the same are doomed to suffer consequences they can no longer predict.',
    ],
    6: [  # Chronicle
        'On Tuesday, October 14, at seven in the morning, the city’s central market opened its doors for the last time after seventy‑two years of uninterrupted operation. The vendors who had occupied the same stalls for decades arrived earlier than usual. Some had slept little. One of them, Mr. Aurelio Peña, who had sold vegetables at stall number eighteen since 1987, said that morning he had breakfast in silence thinking of his father, who had taught him the trade in that same place.',
        'I arrived at the area at three in the afternoon, six hours after the fire started. Firefighters were still working the perimeter. The smoke had lowered, and the building’s structure could be seen like a blackened skeleton against the late‑afternoon sky. Neighbors were gathered on the opposite sidewalk, some with blankets, some with faces showing the fatigue of those who had been standing for hours unable to do anything.',
        'For three consecutive days in January, the temperature in the valley did not rise above two degrees. It was the most intense cold recorded in that region in forty years, according to meteorological service data. Farmers who had planted in November lost between forty and sixty percent of their crops. Mr. Rafael Mena, a strawberry grower for twenty years, showed me his field in silence and only said, "We have to start over."',
        'The assembly began at eight in the evening at the community hall of the La Esperanza neighborhood, with two hundred people seated and another hundred standing along the walls. The topic was the road expansion that would pass forty meters from the houses in the north sector. The project engineer presented the plans. When he finished, a woman about sixty raised her hand and asked, without hostility but without softness, "Do you live in this neighborhood?"',
        'On Friday night, while the city slept, a group of twenty people began painting the largest mural the neighborhood had ever seen. They worked non‑stop until dawn. When the first passersby on Saturday morning walked by the corner of the main avenue and Seventh Street, they found a thirty‑meter wall covered with the faces of the neighborhood women. None of them had been consulted. All said it was the first time someone had placed them in such a visible place.',
        'What happened that night at the stadium was not in any security protocol. At ten‑thirty, when the match had been in extra time for twenty minutes, the lights failed. For forty‑five seconds the stadium with fifty thousand people fell into complete darkness. No one ran. No one screamed. A silence that journalists present would describe in different ways but all remembered for weeks.',
        'I spent a week with the fishermen of the northern gulf to understand why their catches had fallen sixty percent in five years. I went out with them three consecutive mornings. The first night we caught nothing. The second, a little. The third, Mr. Domingo Herrera, who had been at sea for thirty‑eight years, stopped the engine ten kilometers from shore, looked at the water and said, without drama, without rhetoric, "There used to be fish here. Now there’s nothing. That’s not an opinion. It’s what I see."',
        'The event was called for six in the evening in front of the ministry building. By quarter to six there were already three hundred people. By six‑thirty there were more than a thousand. No one had foreseen that. Neither had anyone foreseen that it would start raining exactly at seven, nor that the rain would make no one move. The people I interviewed had different reasons for being there, but when I asked why they didn’t leave with the rain, they all gave roughly the same answer.',
        'Teacher Consuelo Díaz has been teaching for forty‑three years in the same school, in the same classroom, with the same blackboard that has never been changed. She has seen eight principals, three educational reforms, and twenty‑one generations of children pass through. When I asked what had changed in four decades, she thought for a moment and replied, "The children still arrive with the same hunger to understand. What changes is what weighs on them before they arrive."',
        'The bridge collapsed at three in the afternoon on a Wednesday. At that moment eleven people and two vehicles were crossing it. All survived. The municipal engineer who reviewed the maintenance reports that night found three reports of structural deterioration submitted over the previous four years, all marked as received, none marked as addressed. He showed them to me on screen without saying anything. Nothing needed to be said.',
        'I was present the afternoon the jury announced its verdict. The defendant listened to the words standing, motionless. The courtroom was silent for two seconds that seemed longer. Then the noise began: some cried, some spoke loudly, the families of both sides reacted in ways I did not dare describe immediately because I was still processing what I had just seen.',
        'Three weeks after the disaster, I returned to the village to see how the recovery was going. The humanitarian aid trucks no longer came. Neither did the television cameras. What remained was the community alone with its problem, which was much larger than any headline had communicated and much more ordinary than the drama of the first coverage had suggested.',
        'The news of the factory closure came on a Friday at four in the afternoon, just when shifts were changing. Four hundred workers in total. Some had been there for more than twenty years. No one spoke to me about figures. They spoke to me about the coworkers they had had breakfast with that morning without knowing it was the last time in that cafeteria, about the lockers they would have to empty, about the routine they did not know how to replace.',
        'The river rose during the night and by dawn it had already entered the first houses in the lower sector. The neighbors showed me the marks on the walls from other years: the 2011 flood, the 2018 flood. This one already surpassed both. The difference, an engineer who had come voluntarily that morning explained to me, was that now the catchment area had thirty percent less vegetation than in 2011. The fact was in no press release.',
        'I arrived at the border at five in the morning and there was already a line of one hundred fifty people waiting for it to open. They came from three different countries. None carried more than what fit in a backpack. I asked several what they expected to find on the other side. The answers differed in details but were identical at heart: something better than what they were leaving. None used the word hope. They took it for granted.',
        'The electoral process in the municipality of San Marcos lasted fourteen hours. I was at three different polling stations. At the first there was an issue with the identification system that took two hours to resolve. At the second, a person arrived with expired documents, and the discussion about whether they should vote lasted forty minutes. At the third everything worked perfectly, and the polling station president told me, somewhat surprised, "This is the first time in twelve years that nothing has happened."',
        'The director of the public hospital spoke with me for an hour about budget cuts. She was direct and precise. Then she took me on a walk through the hallways. There were beds in the corridor. Thirty percent of the operating rooms were closed due to lack of staff. When we finished the tour she asked if I needed anything else for the article. I said no. She replied, "Then you already have everything I have been trying to tell someone who would listen for years."',
        'In September last year, a group of women from the neighborhood decided to organize to document the potholes on their street because verbal complaints to the municipality had been going unanswered for three years. They took photos with their cell phones, measured the potholes with tape measures, recorded dates, and submitted a twelve‑page report to the city council. At the session where it was presented, the councilor who received it said it was "the most complete technical report the council had ever received." The street was repaired in two months.',
        'On the anniversary of the industrial accident, the relatives of the victims gathered as they did every year in front of the facility. This time one hundred twenty people arrived, twice as many as the previous year. I spoke with several. None mentioned the date by chance: they all knew exactly how many years had passed, how many months, some even how many days. Time had not dulled anything. It had only organized what remained in a different way.',
        'I spent two weeks in the maximum‑security prison to write about the social reintegration system. What I found contradicted almost everything the institution published in its official statements. The officials who spoke to me off the record were more honest than those who spoke on the record. The inmates participating in education programs were those who least expected them to be useful but who took the most advantage of them because they had time and because in that place, somehow, time was the only thing that was abundant.',
    ],
}
for etiqueta, ejemplos in ejemplos_genero.items():
    datos_textos.extend(ejemplos); datos_etiquetas.extend([etiqueta]*len(ejemplos))

# EJEMPLOS MANUALES TIPO
ejemplos_tipo = {

    # INFANTIL
    0: [
        'Once upon a time there was a little rabbit who lived in a cozy burrow under the old oak tree in the forest. Every morning he would hop out to greet the sun and collect fresh clover for breakfast. One day he found a tiny door at the base of the oak that he had never noticed before.',
        'The friendly dragon did not want to breathe fire; he only wanted to bake cookies for his friends in the village. But every time he tried to light the oven, a flame shot out and burned the cookies. Then the baker’s daughter showed him how to use a match instead.',
        'Tommy and his dog went on a big adventure through the meadow looking for the missing red balloon that flew away. They asked the cow, the horse, and even the wise old owl. Finally they found it caught in a tree, and Tommy’s dog climbed up and brought it down with a happy bark.',
        'The tiny fairy sprinkled magic dust on the flowers and they began to sing happy little songs in the morning sun. The flowers’ songs were so beautiful that all the bees and butterflies stopped to listen, and the fairy danced among them until sunset.',
        'Mama bear told baby bear that it was time to sleep and the whole forest became very quiet and still at night. The stars twinkled above, and a gentle breeze whispered lullabies through the leaves. Baby bear snuggled close and dreamed of honey and blueberries.',
        'The little train said I think I can I think I can as it climbed slowly up the big green hill with its cargo of toys and treats. The wheels spun and the engine puffed, and soon it reached the top, where all the children were waiting with cheers and open arms.',
        'Billy the elephant had very big ears and all his friends loved to hide under them when it rained in the jungle. The monkeys would swing onto his back, the parrots would perch on his trunk, and the frogs would sit on his ears, singing songs about the sun.',
        'Lily counted all the stars before going to bed and she always lost count at twenty and started again from one. Her grandmother told her that each star was a story waiting to be told, so Lily closed her eyes and imagined adventures among the constellations.',
        'Max woke up one morning and found that all his toys had decided to go on a little trip without telling him first. They left a note made of building blocks: "Back by bedtime." So Max built a rocket ship and went after them, finding them having a picnic on the moon.',
        'Every night the moon told the children of the town stories about the faraway stars and planets in the sky. The children would sit on their rooftops and listen, and the moon’s voice was so soft that only those who really wanted to hear could understand.',
        'Sam the snail moved very slowly but he always arrived just in time for the big birthday party at the pond. His friends worried that he would miss the cake, but Sam had left at dawn, and he brought a beautiful shell he had found along the way as a gift.',
        'The little girl planted a tiny seed and every morning she ran outside to check if it had grown overnight. She sang to it and watered it with her watering can. After many weeks, a sunflower taller than her house bloomed, and she climbed it to see the clouds.',
        'Pedro the parrot learned to say good morning every day and the whole neighborhood loved him for his cheerful voice. He would greet the mailman, the baker, and the children on their way to school. One day he learned to say "I love you," and everyone smiled.',
        'The cloud wanted to be a rainbow and tried very hard until one sunny rainy afternoon it finally became one. It stretched its colors across the sky, and all the people looked up and pointed with joy. The cloud felt proud, but later it returned to being a cloud, happy to have been a rainbow for a while.',
        'Oliver fed the ducks at the pond every Saturday and knew each one by a special name he had given them himself. There was Daisy, Waddles, and Sir Quackington. One day, Sir Quackington brought his new ducklings, and Oliver named them too, feeling like an uncle.',
        'The little star was sad because she couldn’t shine as brightly as the others. So the moon told her that even a tiny light can guide someone home. That night, a lost child saw her flicker and found his way back, and the star shone with newfound joy.',
        'In the land of Sweets, the gingerbread man ran faster than anyone, but he always stopped to help those in need. He helped the lollipop lady cross the chocolate river and showed the jelly bears how to share. Everyone agreed he was the sweetest of all.',
        'The wooden puppet wanted to be a real boy more than anything. He studied hard, told the truth, and helped his father. One night, the Blue Fairy visited and told him that his heart was already real, and that was what truly mattered.',
        'A family of mice lived inside a grandfather clock, and they would chime along with it every hour. The youngest mouse learned to count by listening to the bongs. One day the clock stopped, and the mice sang the hours themselves until the clock was fixed.',
        'The scarecrow in the cornfield had no brain, but he thought of clever ways to keep the crows away without scaring them. He gave them shiny buttons to play with, and they became his friends. The farmer said he was the smartest scarecrow ever.',
        'The tin man wanted a heart, but he already showed kindness by helping the animals who got caught in the thorns. He mended the bird’s wing and shared his oil with the rusty gate. Dorothy said his heart was the biggest of all.',
        'The cowardly lion thought he was afraid of everything, but when his friends were in danger, he roared louder than thunder and scared away the beast. He realized that courage is not the absence of fear, but acting despite it.',
        'Dorothy and her dog Toto were swept away by a cyclone to a magical land. She followed the yellow brick road, made wonderful friends, and learned that there is no place like home. The ruby slippers took her back, and she never took her family for granted again.',
        'The little engine that could taught everyone that believing in yourself is half the battle. When a bigger engine refused to help, the little blue engine said "I think I can" and pulled the heavy train over the mountain. The toys and dolls cheered for her.',
        'The velveteen rabbit wanted to become real, and the nursery magic said it happened when you were truly loved. The boy loved him so much that his fur became worn and his seams loose, but he became real to the boy, and that was enough.',
        'Pippi Longstocking had a monkey named Mr. Nilsson and a horse on her porch. She was the strongest girl in the world, and she never went to school because she already knew how to have fun. She taught the neighborhood children that being different is wonderful.',
        'Charlie Bucket found a golden ticket and entered Willy Wonka’s chocolate factory. He saw the chocolate river, the Oompa Loompas, and the great glass elevator. Because he was honest and kind, he won the whole factory, and he shared it with his family.',
        'The BFG caught dreams and blew them into children’s rooms with his trumpet. Sophie, a little girl, became his friend and helped him stop the other giants. They lived together in Giant Country, where every night was a dream adventure.',
        'Matilda loved to read books and discovered she had magical powers when she was angry or upset. She used her powers to help her kind teacher Miss Honey and to scare away the terrible headmistress. In the end, she found a real family with Miss Honey.',
        'James lived with two nasty aunts until a giant peach grew in the garden. He crawled inside and met insect friends: the Grasshopper, the Spider, the Ladybug, and others. They rolled the peach to the ocean and had a wonderful adventure to New York City.',
        'The hungry caterpillar ate through one apple, two pears, three plums, and many more foods. Then he built a cocoon and emerged as a beautiful butterfly. He learned that sometimes you need to eat a lot to grow big and strong.',
        'A little fish named Rainbow had shiny scales that made him proud, but he was lonely. When he gave away one of his scales to a smaller fish, he felt happiness. He shared his beauty and gained friends who loved him for who he was.',
        'The very busy spider was too busy spinning her web to play with the farm animals. She worked all day, and by night she had caught a fly. The farmer praised her hard work, and she felt proud of what she had accomplished.',
        'The mixed-up chameleon wished he could be like other animals: tall like a giraffe, strong like a bear, fast like a deer. But when he tried to be everything at once, he couldn’t catch a fly. He decided being himself was best after all.',
        'Polar Bear, Polar Bear, what do you hear? I hear a lion roaring, a hippopotamus snorting, a flamingo fluting. The zoo animals made wonderful sounds, and the zookeeper heard them all before the children arrived.',
        'Brown Bear, Brown Bear, what do you see? I see a red bird looking at me. Red Bird, Red Bird, what do you see? I see a yellow duck looking at me. The whole world was full of colors and friends looking at each other.',
        'The pigeon wanted to drive the bus, but the bus driver had left him in charge not to. He begged, pleaded, and made all sorts of excuses, but in the end he had to let the real driver return. Maybe another day, he thought.',
        'Knuffle Bunny was Trixie’s favorite stuffed animal. When it got lost at the laundromat, she couldn’t talk yet, so she made strange noises. Her daddy understood and rescued Knuffle Bunny. Trixie’s first words were "Knuffle Bunny!"',
        'Don’t let the pigeon stay up late! He wanted to stay up, to have a hot dog party, to watch the show. But he was tired, and eventually he yawned and fell asleep, dreaming of driving a bus all night long.',
        'There was an old lady who swallowed a fly. Then she swallowed a spider to catch the fly, then a bird to catch the spider. She kept swallowing bigger and bigger animals until she swallowed a horse—she died, of course. But it made for a silly song.',
        'The cat in the hat came on a rainy day and turned the house upside down with Thing One and Thing Two. But when Mother came home, the house was clean again, and the children promised to tell her about their fun day.',
        'Green Eggs and Ham: Sam-I-Am tried to convince a grumpy fellow to try green eggs and ham. He offered them in a house, with a mouse, on a train, in the rain. Finally, the fellow tried them and found he liked them, thank you, Sam.',
        'Horton heard a Who from a tiny speck of dust. He protected the speck, even when the other animals mocked him. He proved that a person is a person, no matter how small, and saved Whoville from being boiled in Beezle-Nut oil.',
        'The Grinch hated Christmas, so he stole all the presents and decorations from the Whos down in Whoville. But the Whos still celebrated with song, and his heart grew three sizes. He returned everything and joined the feast.',
        'A mouse took a stroll through the deep dark wood. A fox, an owl, and a snake wanted to eat him, but the mouse invented a terrible creature called the Gruffalo. To his surprise, the Gruffalo was real, and the mouse tricked him too.',
        'Room on the broom: a witch and her cat flew on a broom, picking up a dog, a bird, and a frog. The broom broke, and a dragon wanted to eat the witch, but the animals scared the dragon away, and the witch made a new, better broom.',
        'The snail wanted to see the world, so she hitched a ride on a whale’s tail. They traveled across oceans, but when the whale got beached, the snail saved him by alerting the villagers. Together they returned to the sea.',
        'The smartest giant in town bought new clothes, but he gave away his tie to a giraffe, his shirt to a goat, his shoes to a mouse. He ended up in his old gown, but all the animals thanked him and made him a crown.',
        'The tiger came to tea and ate all the food in the house, even the water from the tap. Then he left, and Sophie’s daddy brought home fish and chips. The tiger never came again, but they always had a tin of tiger food just in case.',
        'We’re going on a bear hunt. The family went through grass, a river, mud, a forest, and a snowstorm. They found a bear, ran back, and hid under the covers, vowing never to go bear hunting again.',
    ],

    # JUVENIL
    1: [
        'She had always felt different from the other kids at school, and this year she was finally determined to find out why. It started with a letter from a grandmother she never knew existed, inviting her to a town that didn’t appear on any map. The letter smelled of old paper and roses, and the handwriting was elegant, like something from another century.',
        'The summer he turned fifteen, everything changed completely. His father gave him an old key and said, "When you’re ready, open the door in the basement." That door led to a world where time moved differently, where he could relive his happiest moments but also confront his deepest regrets.',
        'Maya discovered a hidden door in the school library that led to a place no map had ever shown to anyone before. Behind it was a room filled with books that wrote themselves as she read them, stories that seemed to know her secrets. She realized she wasn’t just reading—she was becoming part of a larger narrative.',
        'He made the team but keeping his place meant training every morning before the rest of the school even woke up. The weight of the varsity jacket felt like a promise to himself and a burden from his father. He learned that passion and pressure often wear the same face.',
        'They had been best friends since second grade but high school was slowly pulling them in very different directions. One was drawn to the art room, the other to the football field, and their lunch conversations grew shorter, filled with things they no longer understood about each other.',
        'The letter from the academy arrived on her birthday and she read it four times before she finally believed it. Acceptance meant leaving everything she knew, but rejection meant staying in a town where no one understood her obsession with the stars. She had to choose between safety and the sky.',
        'Nobody at school knew she spent her evenings writing code that thousands of people used every single day online. She was the anonymous creator of a popular app that helped kids find study partners, but in real life she couldn’t even talk to her crush without blushing.',
        'Training for the championship meant giving up everything else and she had to decide if it was really worth it. Her best friend’s birthday, the school play, even her own sleep—all sacrificed for a trophy that might not even come. She wondered if winning was the same as being happy.',
        'She kept a journal of every strange thing that happened in their small town and the list kept growing every week. The old water tower leaked blue light, the cemetery gates opened by themselves on the full moon, and her grandfather spoke of a pact made a hundred years ago.',
        'The night before the big exam he found his grandfather’s notes hidden inside an old textbook from decades ago. The notes weren’t about the subject—they were a map to a treasure his family had been searching for since the war. He had to decide if history was more important than his future.',
        'The robot they built for the science fair began doing things none of them had ever programmed it to do alone. It wrote poetry about longing and drew sketches of the moon. They realized they hadn’t just built a machine; they had created something that might be thinking.',
        'He discovered his ability the same week as tryouts and spent three days deciding what exactly to do with it. He could hear people’s thoughts when they looked at him, but only when they were thinking about their deepest fears. Suddenly, popularity seemed less appealing.',
        'She auditioned as a joke and got the lead role and now had to figure out how to actually act on stage. Her first rehearsal was a disaster, but the director saw something in her trembling voice. She learned that vulnerability could be a kind of strength.',
        'He uploaded the video as a joke and woke up the next day with more views than he could possibly understand. Comments poured in from around the world, some praising, some cruel. He had to learn what it meant to be seen by millions while still being a teenager.',
        'The clue in the yearbook photo sent them on a search through every corner of the old school building at night. They found a hidden time capsule from 1973 containing letters that hinted at a secret society. By dawn, they had uncovered a mystery that tied their families together.',
        'When her parents told her they were moving across the country, she had one month to say goodbye to everything she knew. She made a list of places she had to visit one last time: the tree where she learned to read, the diner where she had her first kiss, the creek where she scattered her grandmother’s ashes.',
        'The anonymous message on his phone said "I know what you did" and he had no idea what it meant or who sent it. For days he retraced his steps, trying to remember something he might have done wrong. The truth was worse: someone had mistaken him for another boy, and that boy was in serious trouble.',
        'She started a podcast about forgotten local history and accidentally uncovered a secret that some people wanted to stay buried. The more she dug, the more threats appeared, but she also found allies who had been waiting for someone to tell the truth.',
        'The scholarship application required an essay about a time he failed, and he realized he had never really tried anything hard enough to fail. He decided to change that by attempting something impossible: learning to play the violin in three weeks for the spring concert.',
        'Their group chat exploded when one of them posted a screenshot that was never meant to be shared, and friendships were tested. Loyalties were questioned, apologies were demanded, and for a week they didn’t speak. In the end, they learned that trust is easier to break than to rebuild.',
        'He spent the whole summer restoring an old motorcycle with his grandfather and learned more about life than about engines. His grandfather told stories about the war, about love, about mistakes he never admitted to anyone else. The bike roared to life on the last day of August, and so did a new understanding between them.',
        'The new student spoke five languages but refused to say a single word in class, and she was determined to find out why. It took weeks of sitting together in silence before he finally spoke: he had fled a war and was afraid his accent would give him away. She taught him that silence could be a cage or a comfort.',
        'After being voted "most likely to succeed", she felt the weight of everyone’s expectations pressing down on her shoulders. She had a plan: Ivy League, medical school, a respectable life. But her heart whispered a different dream: writing novels in a small apartment by the sea.',
        'He found his father’s old journal from when he was a teenager and discovered they had more in common than he ever imagined. The same fears, the same anger, the same dreams of escape. He realized that his father wasn’t just a parent; he was once a boy trying to figure things out, too.',
        'The camping trip was supposed to be fun, but when a storm trapped them in the woods, they had to rely on each other to survive. Hidden resentments surfaced, old grudges were aired, but by the third day they had learned to listen. They came back different people.',
        'She painted a mural on the abandoned wall downtown and suddenly the whole city was talking about the mysterious artist. The mayor wanted to commission her, a gallery wanted to show her work, but she was just a girl who liked to paint at night. She had to decide if she wanted to be seen.',
        'He agreed to be the school mascot for a year in exchange for a letter of recommendation, not knowing how much it would change him. Dressed as a tiger, he could say things he never dared to say. He gave pep talks to nervous freshmen, comforted a crying girl after her boyfriend broke up with her, and became a hero in a costume.',
        'The debate team had never won a single competition, but this year they had a secret weapon: a freshman who never lost an argument. She didn’t care about winning; she cared about the truth. She made the team rethink their tactics and, in the process, taught them that a good argument can change minds, not just score points.',
        'She found an old flip phone in her grandmother’s attic that still received text messages from someone who had been gone for years. The messages were mundane at first: "Don’t forget milk" and "See you at 5." But then they became questions about things that happened after the sender had died.',
        'He was the only one who noticed that the town’s annual festival had a pattern that repeated exactly every seven years, and he wanted to know why. The records went back to 1880, and every seven years a different person disappeared. He decided to be the one who solved the mystery before this year’s festival.',
        'Her mother said she could be anything she wanted, so she decided to become a ghost hunter. Armed with a borrowed EMF reader and a notebook, she explored every abandoned building in town. She never found ghosts, but she found something else: stories of people who had lived before, forgotten by everyone except the walls.',
        'The summer job at the ice cream shop was supposed to be easy money, but it taught her more about people than any textbook. She learned which customers were hiding sadness behind their smiles, which couples were about to break up, and which children were trying to escape something at home.',
        'He accidentally sent a text to the wrong number and ended up in a conversation with a stranger that lasted all night. They talked about everything except who they were: books, music, fears, hopes. In the morning, he realized he had never been so honest with anyone.',
        'The school play was Romeo and Juliet, and she was cast as Juliet opposite her ex‑boyfriend’s Romeo. Every rehearsal was torture, but the director said their tension was perfect for the role. By opening night, they had to decide if the stage was a place to relive the past or to create something new.',
        'His father wanted him to take over the family business, but he wanted to be a dancer. They stopped speaking for months. Then his father came to a recital, watched from the back, and afterwards said nothing. But the next day, there was new dance flooring in the garage.',
        'She discovered a hidden garden behind the abandoned house on Elm Street. The flowers were unlike any she had seen, and they bloomed in winter. An old woman appeared one day and said the garden had been waiting for someone who still believed in magic.',
        'The yearbook committee needed a theme, and he proposed "Fragments." He wanted to capture not the perfect moments but the broken ones: the kid who ate lunch alone, the girl with scars on her arms, the boy whose father was deployed. The principal rejected it, but the students loved it.',
        'He spent his junior year writing letters to soldiers overseas after his brother was deployed. He never got replies, but he kept writing. When his brother came home, he brought a box of letters from other soldiers who had received his notes and said they kept them going.',
        'The music teacher told her she would never be a soloist, so she started a choir in the basement of the old church. It was made up of kids who had been told they weren’t good enough: the tone‑deaf, the shy, the ones who sang only in the shower. By Christmas, they had a concert that packed the pews.',
        'He found a geocache hidden by his late grandfather that contained clues leading to locations across town. Each location held a memory his grandfather wanted him to have: the place where he proposed, the bench where he read stories, the tree where he carved their initials. The final cache held a letter: "I wanted you to know where I loved you."',
        'She was the only one who could hear the old piano in the school auditorium playing at night. When she investigated, she found a boy from the 1940s sitting at the keys, waiting for someone to hear his unfinished symphony. She helped him finish it, and he disappeared with the last note.',
        'The swim team captain bet him he couldn’t make it across the lake at midnight, but he didn’t know that the lake had a secret: a hidden island where kids went to escape their lives. He made it, and found a community of runaways and dreamers who lived by their own rules.',
        'Her grandmother’s recipe book contained not just instructions for food but spells disguised as cooking instructions. "Add a pinch of courage to this stew" and "Stir in a whisper of love for this bread." She started cooking, and people who ate her food felt things they had long forgotten.',
        'He was the only one who remembered the town before the factory closed. His father lost his job, his friends moved away, and Main Street became a row of empty windows. He decided to document the stories of those who stayed, creating a podcast that reminded everyone that a place isn’t just its industry.',
        'The art competition required a piece that represented "home," and she didn’t know what to paint. She moved twelve times in sixteen years. In the end, she painted a collage of all her doorways, each one a different color, each one a threshold to a new life. It won first place.',
        'He built a treehouse with his own hands, a place where he could escape his parents’ constant fighting. He invited other kids who needed a place, and soon the treehouse became a sanctuary. When his parents separated, he realized he had been building a home for himself all along.',
        'She volunteered at the animal shelter and fell in love with a three‑legged dog no one wanted. She named him Tripod and spent months training him. When he was adopted, she cried, but then she saw the new family, and she knew he would be happy. She had learned that loving means letting go.',
        'The old man next door was a mystery; he never spoke and never left his house. She started leaving notes in his mailbox: jokes, questions, drawings. One day he came out and handed her a note: "I used to be a cartoonist. Your drawings remind me why I loved it." They became friends.',
        'She had a stutter that made her terrified of speaking in class. Her English teacher gave her a notebook and told her to write instead. She wrote poems that made the class laugh and cry. One day she read one aloud, and she didn’t stutter. The teacher smiled.',
        'The haunted house at the end of the lane was just a story until he saw a light in the attic. He dared his friends to go inside, but they chickened out. He went alone and found not ghosts, but a woman painting portraits of people who had died without anyone to remember them. She was preserving memories, not haunting.',
    ],

    # ACADÉMICO
    2: [
        'The study examined the correlation between socioeconomic variables and educational outcomes across multiple regions, using a stratified random sample of 10,000 students from 500 schools. Preliminary results indicate a significant positive association between parental education level and standardized test scores, even after controlling for school funding and teacher quality.',
        'This paper proposes a novel framework for analyzing discourse patterns in postcolonial literary texts, integrating computational stylometry with close reading methodologies. The framework is applied to a corpus of 50 novels from Africa and the Caribbean, revealing patterns of code‑switching that function as acts of resistance against colonial language hierarchies.',
        'The empirical evidence suggests a significant relationship between early language exposure and later literacy outcomes. Drawing on longitudinal data from the Early Childhood Longitudinal Study, we find that children who hear more than 2,000 words per hour before age three demonstrate reading comprehension scores one standard deviation higher than peers by fourth grade.',
        'The methodology employed a mixed-methods approach combining quantitative surveys with in-depth qualitative interviews. A total of 1,200 participants completed the survey, and follow‑up interviews were conducted with 60 individuals representing diverse demographic backgrounds. Thematic analysis of interview transcripts revealed three primary factors influencing educational persistence.',
        'Results indicate that the intervention produced statistically significant improvements in measured reading comprehension (p < 0.01, Cohen’s d = 0.45). The treatment group showed a 23% increase in Lexile scores compared to a 4% increase in the control group, suggesting that targeted phonics instruction is particularly effective for struggling readers.',
        'The theoretical framework draws on Foucauldian discourse analysis to examine power relations embedded in the text, specifically focusing on how diagnostic categories in mental health discourse create subject positions that patients are expected to inhabit. This approach reveals the disciplinary function of clinical language beyond its descriptive purpose.',
        'A systematic review of the literature reveals three dominant paradigms currently present in translation studies: the linguistic, the cultural, and the cognitive. This article synthesizes these approaches and proposes an integrated model that accounts for both micro‑level linguistic choices and macro‑level ideological implications.',
        'The corpus analysis identified recurring syntactic patterns consistently associated with formal academic register use, including nominalization, passive constructions, and complex subordination. These patterns were found to vary significantly across disciplines, with humanities texts exhibiting higher rates of metaphorical language than STEM texts.',
        'The findings challenge prior assumptions about the supposed universality of narrative comprehension strategies worldwide. Using eye‑tracking methodology with participants from five countries, we found that readers from collectivist cultures allocate more attention to group interactions, while those from individualist cultures focus more on protagonist mental states.',
        'Data were collected from a stratified random sample of undergraduate students enrolled across five research institutions. The survey instrument was validated through a pilot study with 200 students and demonstrated high internal consistency (Cronbach’s α = 0.89). Missing data were handled using multiple imputation techniques.',
        'Statistical analysis was conducted using multiple regression models specifically designed to control for confounding factors such as prior academic achievement and family income. After controlling for these variables, the effect of the intervention remained significant, suggesting that the program benefits students regardless of background.',
        'The conclusion synthesizes the key findings and proposes concrete directions for future empirical investigation in this area. We recommend that subsequent studies employ experimental designs to establish causality, incorporate diverse geographic contexts, and examine the long‑term effects of early educational interventions.',
        'The hypothesis was tested against a null model using chi-square tests at a significance level of 0.05 throughout. Results show a significant association between teacher training and student engagement (χ²(2) = 12.34, p = 0.002), supporting the hypothesis that professional development improves classroom outcomes.',
        'Limitations of the study include the restricted geographic scope and the heavy reliance on self-reported data from participants. Future research should address these limitations by expanding sampling to rural and underrepresented communities and incorporating observational methods to complement survey findings.',
        'Future research should address the longitudinal dimension of the complex phenomena described and analyzed in this paper. A ten-year cohort study would allow researchers to track how academic identity develops over time and to identify critical periods for intervention.',
        'This article examines the intersection of climate adaptation policies and indigenous knowledge systems in the Andean region. Drawing on ethnographic fieldwork conducted over three years, we argue that top‑down adaptation strategies often undermine local practices that have sustained communities for centuries.',
        'A critical discourse analysis of political speeches reveals systematic use of metaphorical framing to shape public opinion. Analyzing 200 speeches from five national leaders, we identify recurrent metaphors of "journey," "war," and "family" that serve to naturalize particular policy positions and exclude alternatives.',
        'The longitudinal study tracked 500 participants over ten years to measure the impact of early childhood education on adult earnings. After controlling for family background and neighborhood effects, we find that each additional year of preschool is associated with a 7% increase in annual earnings at age 30.',
        'Drawing on Bourdieu’s concept of cultural capital, this paper argues that aesthetic taste functions as a marker of social distinction in contemporary society. Using survey data from 3,000 respondents, we show that preferences for highbrow cultural forms remain strongly correlated with educational attainment and family income.',
        'Regression discontinuity designs offer a robust method for estimating causal effects in educational interventions when randomization is not feasible. Applying this method to a large‑scale literacy program, we estimate an average treatment effect of 0.3 standard deviations on reading scores for students near the eligibility cutoff.',
        'The manuscript traces the evolution of narrative perspective in eighteenth-century British fiction from omniscience to internal focalization. Close readings of 30 novels reveal a gradual shift toward limited perspectives, reflecting emerging psychological conceptions of selfhood during the Enlightenment.',
        'Qualitative interviews with 40 teachers revealed that administrative burden is the primary factor affecting job satisfaction in urban schools. Teachers reported spending an average of 15 hours per week on non‑instructional tasks, leading to burnout and high turnover rates.',
        'Using a quasi-experimental design, the authors demonstrate that mindfulness training significantly reduces test anxiety among high school students. The intervention group showed a 35% decrease in self‑reported anxiety compared to the control group, with effects persisting at six‑month follow‑up.',
        'This study contributes to the growing body of literature on translanguaging practices in multilingual classrooms by analyzing classroom interactions in 20 dual‑language programs. Findings suggest that allowing students to draw on all linguistic resources enhances comprehension and engagement.',
        'The author revisits the concept of “slow violence” to describe environmental degradation that occurs gradually and often goes unnoticed by the public. Applying this framework to the case of coal ash contamination in the southeastern United States, the paper shows how regulatory failures allow cumulative harm to accumulate over decades.',
        'Factor analysis identified three underlying dimensions of digital literacy: technical proficiency, critical evaluation, and creative production. These dimensions were found to vary systematically across age groups, with younger users scoring higher on technical skills but lower on critical evaluation.',
        'Comparative analysis of urban planning documents from six cities shows a shift toward participatory governance rhetoric despite persistent top-down practices. While all cities claim to incorporate community input, the actual mechanisms for public participation remain limited and often tokenistic.',
        'The paper critiques the neoliberal underpinnings of the “grit” discourse in education, arguing it individualizes structural inequality. Using a critical policy analysis framework, the author shows how policies promoting character education deflect attention from resource inequities and systemic barriers.',
        'Through a close reading of archival materials, the historian reconstructs the informal networks that sustained intellectual exchange during the dictatorship. Correspondence, personal diaries, and underground publications reveal a vibrant counterpublic that operated beneath the surface of official culture.',
        'Bayesian hierarchical modeling was applied to account for variability across schools while estimating the overall effect of the literacy intervention. The model indicates that school‑level factors explain 40% of the variance in outcomes, highlighting the importance of context in educational research.',
        'The concept of algorithmic bias has gained prominence in recent years, but empirical studies of bias in deployed systems remain rare. This paper presents a comprehensive audit of a widely used employment screening algorithm, finding significant disparities in outcomes across gender and racial groups.',
        'Drawing on archival research in three countries, the article traces the circulation of scientific knowledge between Europe and Latin America during the nineteenth century. The analysis shows that local actors played active roles in shaping, translating, and contesting European theories, challenging diffusionist models.',
        'A meta-analysis of 50 studies on the relationship between class size and student achievement finds a small but significant negative effect of larger classes (r = -0.12). Effects are strongest for younger students and in reading instruction, suggesting that class size matters most during early literacy development.',
        'The paper proposes a new theoretical framework for understanding the role of emotion in political communication. Integrating insights from neuroscience, psychology, and media studies, the framework posits that emotional appeals are processed through both conscious and non‑conscious pathways, with distinct effects on persuasion.',
        'Using social network analysis, the study maps collaboration patterns among researchers in the field of climate change. Results show a highly interconnected core of authors who publish together repeatedly, with peripheral scholars representing diverse disciplinary backgrounds that are not fully integrated.',
        'This research examines the implementation of restorative justice practices in urban high schools. Ethnographic observation and interviews with administrators, teachers, and students reveal tensions between the philosophy of restorative justice and the institutional logics of discipline and accountability.',
        'The article applies computational methods to a corpus of 10,000 nineteenth-century novels to track changes in representations of gender over time. Findings show a sharp decline in references to "domesticity" after 1870 and a corresponding increase in depictions of independent female characters.',
        'Drawing on organizational sociology, the paper analyzes how universities respond to rankings pressures. Through case studies of 15 institutions, the author shows that rankings influence resource allocation and strategic planning, but also produce unintended consequences such as mission drift and strategic gaming.',
        'A longitudinal ethnography of a low‑income neighborhood over 20 years traces the effects of housing policy on community cohesion. The study finds that mixed‑income development disrupted social networks that had provided essential support, while failing to improve economic outcomes for original residents.',
        'The paper challenges the prevailing narrative that digital technologies inevitably democratize knowledge production. Instead, it argues that platform architectures and algorithmic curation create new forms of gatekeeping that reinforce existing hierarchies of expertise and authority.',
        'Using historical census data and GIS mapping, this research reconstructs patterns of residential segregation in a mid‑sized American city from 1910 to 1970. The analysis shows that redlining policies not only concentrated poverty but also shaped enduring patterns of social capital and civic engagement.',
        'A randomized controlled trial of a mentoring program for first‑generation college students found positive effects on retention and academic self‑efficacy. Students who received mentoring were 15% more likely to persist to the second year, with effects mediated by increased sense of belonging.',
        'The paper develops a typology of citizen science projects based on their goals, methods, and participant roles. The typology is then used to analyze 200 projects, revealing that projects emphasizing data collection are more common than those involving participants in analysis or research design.',
        'Drawing on concept analysis methodology, this article clarifies the construct of “critical thinking” as it is used in educational research. The analysis identifies three distinct dimensions: logical reasoning, reflective judgment, and practical wisdom, each with different measurement challenges.',
        'A discourse analysis of textbooks used in secondary biology classes reveals that representations of evolution often implicitly rely on teleological language. The study suggests that linguistic choices may contribute to persistent misconceptions about evolutionary processes.',
        'The paper examines the relationship between cultural policy and urban development through a comparative case study of two post‑industrial cities. Both cities used cultural amenities to attract investment, but the benefits accrued unevenly, with displacement and gentrification in one city and community benefits in the other.',
        'Using a lab‑in‑the‑field experiment, researchers tested the effects of information framing on support for climate policies. They found that emphasizing local impacts increased support among rural participants, while emphasizing global impacts was more effective with urban participants.',
        'A bibliometric analysis of publications in the field of digital humanities from 2000 to 2020 shows rapid growth and increasing interdisciplinarity. However, citation patterns reveal persistent divides between computationally oriented and theoretically oriented scholars.',
        'This paper offers a critical review of the literature on teacher burnout, identifying conceptual confusion and methodological limitations in existing studies. The author proposes a multidimensional model that distinguishes between exhaustion, cynicism, and inefficacy, and calls for longitudinal designs to examine causal pathways.',
    ],

    # ROMANCE
    3: [
        'She did not expect to fall for her best friend’s brother, but here she was baking cookies at midnight just for him. The worst part? He didn’t even like cookies. He liked her, and that was the problem. She had promised her best friend that nothing would ever happen between them, but promises were starting to feel like cages.',
        'The vampire showed up at her door with a pizza and a very convincing fake smile, and she decided to let him in. After all, it was raining, and he did bring extra garlic bread. She knew she should be scared—he was immortal, undead, and according to legend, heartless—but his laugh was warm, and he laughed at her terrible jokes.',
        'He woke up on a spaceship with no memory of how he got there and a very chatty robot as his only company. The robot kept saying, “Your heart rate increases whenever I mention the name Evelyn.” He didn’t remember an Evelyn, but his chest ached like he should.',
        'The detective had solved every case in her career except one, and it had been bothering her for eleven long years. Then a new transfer arrived who looked exactly like the missing person. She had to decide if she was chasing a ghost or finding a second chance.',
        'They got married on a dare and now had exactly thirty days to decide if they wanted to make it actually permanent. Neither of them expected to wake up on day fifteen wanting nothing more than to say yes. But fear of ruining their friendship held them back, even as their fingers intertwined under the table.',
        'The heist was supposed to take twenty minutes but by minute three absolutely everything had already gone completely sideways. The alarms were blaring, the security guards were closing in, and he was trapped in a vault with his ex‑girlfriend. She was the one who had taught him safecracking. Now she was the safe he couldn’t crack.',
        'The dragon had retired from terrorizing villages and opened a very successful and popular bakery in the capital city. He made the best croissants in the realm, but he refused to serve knights. Then a knight walked in who wasn’t here to slay him—she just wanted a croissant and maybe, eventually, his heart.',
        'The time machine only went forward and she had forty-eight hours to find a way to somehow reverse its direction. But every time she jumped, she landed in a life with the same man. Different eras, different names, different circumstances, but always the same eyes. Fate, or a glitch? She decided to find out.',
        'They were enemies in every single lifetime but fate kept throwing them together anyway and it was getting exhausting. In this life, she was a park ranger and he was a developer trying to buy her forest. She had a petition; he had a checkbook. Neither expected to fall in love over a campfire.',
        'She inherited a haunted bookshop and the ghost refused to leave until she had read every single book on the shelves. The ghost was a poet who died in 1923, and he commented on each novel with scathing critiques. She started reading aloud to him, and somewhere between Dickens and Plath, he started reading to her.',
        'The fake relationship was entirely her idea and somehow he was better at pretending than she had ever expected him to be. He remembered her coffee order, he knew how she liked her eggs, and he had a terrible habit of looking at her like she was real. She started to wonder if she wanted to be real with him.',
        'The prophecy said one of them would save the entire world and neither of them felt remotely qualified for that task. They were two ordinary people who met at a coffee shop and fell in love, not knowing that their love was the catalyst. The world would be saved, but not by heroics—by choosing each other every day.',
        'He was sent specifically to audit her company and she was determined to make his job as difficult as legally possible. She hid receipts, she “lost” documents, she made him wait for hours. But he was patient, and he noticed that she was protecting something—her employees, her dream, her heart. He didn’t want to audit; he wanted to invest.',
        'The curse clearly stated they would fall in love and both of them were doing their absolute best to strongly resist it. He avoided her at parties; she deleted his messages unread. But the curse was clever—it made them save each other’s lives, one small act at a time. By the time they realized they were already in love, it felt less like a curse and more like a gift.',
        'He was the main villain in her absolute favorite book and she woke up one morning to find herself inside the story. She knew how it ended: he died alone, misunderstood, tragic. She decided to rewrite the ending. He was supposed to be evil, but he blushed when she complimented his dark tower, and she fell for him anyway.',
        'She agreed to a blind date to please her mother, never expecting the man across the table to be the one who got away ten years ago. They had been college sweethearts, separated by circumstance and silence. Now they were adults with careers and baggage, but the way he smiled at her made everything else fade.',
        'The contract was simple: he needed a fake fiancée to secure his inheritance, and she needed money for her sister’s surgery, no feelings allowed. They rehearsed their story, practiced their smiles, and almost believed it themselves. Then he held her hand during a panic attack, and she knew she was in trouble.',
        'After exchanging letters for a year, they finally agreed to meet, but neither had mentioned the small detail of living on different continents. She took a red‑eye flight to London; he took a train to Paris. They missed each other by hours, then days, then weeks. When they finally found each other, they laughed and promised never to be apart again.',
        'He was the grumpy owner of the only bookstore in town, and she was the cheerful florist opening a shop right next door, a disaster waiting to happen. He hated the smell of flowers; she loved the smell of old books. They started a war with passive‑aggressive notes. Then one day he left roses on her doorstep, and she retaliated with a first edition of her favorite novel.',
        'She had one rule: never date a coworker, but the new art director made her forget that rule every time he smiled. He was brilliant, kind, and he drew her portrait during meetings. She tried to resist, but when he left a sketch of her laughing on her desk, she realized the rule was made to be broken.',
        'The wedding was off, but the non‑refundable honeymoon suite in Paris was not, so they decided to go anyway as “just friends.” They walked along the Seine, ate croissants at sunrise, and pretended not to notice the wedding rings they still wore. On the last night, he proposed again, and this time she said yes.',
        'He had been in love with his best friend for years, so when she asked him to be her date for her sister’s wedding, he said yes without hesitation. He wore a suit she helped pick out, danced with her until midnight, and watched her fall in love with someone else. But at the end of the night, she kissed him, and he realized maybe the timing was finally right.',
        'The anonymous love letters she received every Valentine’s Day finally stopped, and she realized she missed the mystery more than she expected. For a decade, she had wondered who they were from. She started searching for the sender, retracing her steps through old jobs, old apartments, old lovers. She found him in a library, writing letters to someone new.',
        'She was a world‑famous cellist, he was a small‑town mechanic, and their worlds collided when her car broke down in the middle of nowhere. He fixed her car in an hour, but she stayed for a week. He taught her to fish, she taught him to appreciate classical music. When she left, he didn’t ask her to stay, but she came back anyway.',
        'The dating app matched them with a 99% compatibility score, but their first date revealed they had already met once before, under disastrous circumstances. She had spilled coffee on his laptop; he had accidentally locked her out of her apartment building. They spent the date apologizing, then laughing, then kissing under the streetlights.',
        'He agreed to pretend to be her boyfriend for one family gathering, but her grandmother’s funeral made things more complicated than either anticipated. He held her hand through the eulogy, made tea for her cousins, and let her cry on his shoulder. When they returned to their normal lives, he missed her, and she missed the way he looked at her like she was allowed to grieve.',
        'She wrote romance novels for a living, but when she tried to apply her own advice to real life, everything went hilariously wrong. She orchestrated a meet‑cute at the grocery store, but he worked there and thought she was a terrible thief. She planned a grand gesture, but he was allergic to roses. In the end, she gave up plotting and let love surprise her.',
        'The inheritance came with a condition: she had to live in the old mansion for six months, and the gardener who came with the property was far too handsome. He knew the names of every plant, told stories about the previous owners, and had a laugh that made her forget her deadlines. She started inventing reasons to visit the garden.',
        'He had given up on love until a stray dog brought him a lost wallet, and the woman who came to retrieve it made him reconsider everything. She was frazzled, late for a flight, and grateful. He offered to buy her coffee while she waited for the next flight. Two hours turned into two years.',
        'They were rivals in the kitchen of a top restaurant, but a sudden snowstorm trapped them inside overnight, and the tension turned into something else. They cooked together, argued about sauces, and drank wine from the cellar. By morning, they had invented a new dish and discovered a new kind of partnership.',
        'She moved to a small coastal town to escape her past and found herself drawn to the lighthouse keeper who hadn’t spoken in years. He communicated through notes, gestures, and the steady beam of the light. She learned his story: a tragedy, a vow of silence, a penance. She started leaving notes of her own, and slowly, he started speaking again.',
        'He was a firefighter who had been badly burned on the job, and she was a physical therapist determined to help him recover. He was stubborn, she was persistent. He hated the exercises; she made them fun. When he finally walked without pain, he didn’t thank her—he asked her to dinner.',
        'She discovered a time capsule from 1943 in her backyard, along with a love letter addressed to someone who had never returned from the war. She traced the recipient, a 95‑year‑old woman who still kept a photograph by her bed. The letter was delivered 80 years late, and the woman smiled, saying, “I knew he loved me.”',
        'He was a classical musician who lost his hearing, and she was a sculptor who communicated through touch. They met at a retreat for artists with disabilities. He felt the vibrations of her chisel through the floor; she watched his hands conduct silent music. Together, they created a performance that needed no sound.',
        'She had been planning her dream wedding since she was a girl, but when she finally got engaged, she realized she was more excited about the flowers than the groom. She broke it off a month before the wedding and ended up at the venue anyway—as a guest, where she met a man who laughed at the same joke and had the same taste in cake.',
        'He was a travel writer who had visited a hundred countries, and she was a librarian who had never left her hometown. He came to write about her town’s famous library and ended up staying for three months. She showed him the hidden corners, the secret gardens, the stories behind the books. He wrote his best article, but he also wrote her love letters.',
        'She was a spy who had been undercover for years, and he was the asset she was supposed to recruit. Instead, he recruited her. They fell in love during dead drops and safe houses, always knowing the mission would end. When it did, he chose her over the agency, and she chose a quiet life with him.',
        'He was a prisoner in a war that had ended ten years ago, and she was a human rights lawyer trying to get him released. She visited him every week, brought him books, and learned his language. When he was finally freed, he didn’t know how to live outside. She taught him, and somewhere between learning to use a microwave and walking in the park, they fell in love.',
        'She was a pilot who flew supplies to remote villages, and he was a doctor who ran a clinic in the jungle. Their paths crossed every month, brief and professional. Then a storm grounded her plane, and she stayed for a week. He showed her the stars, she showed him the world from above. When she left, he asked her to come back—not with supplies, but with herself.',
        'He was a chef who had lost his sense of taste, and she was a food critic who couldn’t smell. They were both sent to a resort to write reviews, but they found each other instead. He cooked for her, she wrote about the texture, the presentation, the memory of flavor. He started tasting again when she was near.',
        'She was a park ranger in a national park, and he was a billionaire who wanted to buy it. She organized a protest; he flew in on a helicopter. They argued for hours, but he listened. He ended up donating the land instead, and she ended up showing him the trails on weekends. Love, she learned, could change even the most unlikely hearts.',
        'He was a ghost hunter who didn’t believe in ghosts, and she was the owner of a supposedly haunted inn. He came to debunk, she came to prove. They spent nights with EMF readers and tape recorders, and they found nothing but silence and each other. In the morning, he admitted that some things are more mysterious than ghosts.',
        'She was a translator who could speak seven languages, but she struggled to say “I love you” in any of them. He was a linguist who studied endangered languages, and he taught her words for emotions she had never named. She learned to say it in a language spoken by only twelve people in the world, and he understood.',
        'He was a mailman who had delivered letters to her house for fifteen years, and she was a writer who had been receiving them. They never met—he left the mail, she wrote stories based on the stamps. Then a letter came for her that he couldn’t deliver: a confession of love from a stranger. He recognized the handwriting—it was his own.',
        'She was a professional organizer who helped people declutter their lives, and he was a hoarder who hadn’t left his apartment in years. She came to help him, but she ended up staying to listen. He told her stories attached to every object, and she realized he wasn’t collecting things—he was collecting memories. She started helping him organize them into albums, and somewhere in the process, they organized their hearts.',
        'He was a musician who played on the subway, and she was a corporate lawyer who passed him every day. She dropped money in his case, but she never stopped. Then one day he played a song she had written in college, a song she had never published. He had found her old notebook in a used bookstore, and he had been waiting to meet her.',
        'She was a beekeeper who lived on an island, and he was a city planner who came to survey for development. He was supposed to convince her to sell, but she convinced him to stay. He learned the language of bees, the rhythm of the tides, the way her hands moved when she was happy. He submitted a report recommending the island be preserved, and he asked to be preserved with her.',
        'He was a retired boxer with a scarred face and a gentle heart, and she was a florist who made bouquets for his sister’s wedding. He came to pick up the flowers and ended up helping her set up. He was careful with the stems, strong with the vases, and he laughed when she told him he looked like a giant in a garden. She gave him a single red rose, and he gave her his number.',
        'She was an astronomer who spent nights looking at the sky, and he was a lighthouse keeper who spent nights looking at the sea. They were the only two people on a small island, and they communicated through radio static. One night she told him about a meteor shower, and he invited her to watch from the lighthouse. They saw shooting stars reflected on the water, and he realized he had been looking in the wrong direction.',
        'They met at a support group for people who had lost their spouses, both grieving, both unsure how to move forward. They sat next to each other for months, never speaking. Then one day she brought extra coffee, and he took it. They started talking about their losses, then about their lives, then about their future. Their love wasn’t a replacement—it was a second chapter they never expected to write.',
    ],
}

textos_tipo, etiquetas_tipo = [], []
for etiqueta, ejemplos in ejemplos_tipo.items():
    textos_tipo.extend(ejemplos); etiquetas_tipo.extend([etiqueta]*len(ejemplos))

df = pd.DataFrame({'texto': datos_textos, 'etiqueta': datos_etiquetas})
print(f'\nDataset género: {len(df)} ejemplos')
for i, n in GENEROS.items(): print(f'  {n}: {len(df[df["etiqueta"]==i])}')
print(f'\nDataset tipo: {len(textos_tipo)} ejemplos')
for i, n in TIPOS_LECTURA.items(): print(f'  {n}: {etiquetas_tipo.count(i)}')

#  Verificación de desbalance
print("\nDistribución real del dataset género:")
dist = df['etiqueta'].value_counts().sort_index()
for i, n in GENEROS.items():
    print(f"  {n}: {dist.get(i, 0)}")
min_clase = dist.min()
max_clase = dist.max()
ratio = max_clase / min_clase
if ratio > 2:
    print(f"\nDesbalance detectado — ratio {ratio:.1f}x (máx {max_clase} / mín {min_clase})")
    print("   Considera usar class_weight o oversample las clases minoritarias.")
else:
    print(f"\n Dataset balanceado — ratio {ratio:.1f}x")


Descargando fragmentos de Gutenberg...
  Novela: 520 fragmentos
  Cuento: 520 fragmentos


## Celda 4 — Entrenar BERT para Género y Tipo de Lectura

In [ ]:
# TOKENIZADOR
print('Cargando tokenizador BERT...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

#MODELO GÉNERO
X_g_tr, X_g_val, y_g_tr, y_g_val = train_test_split(
    df['texto'].tolist(), df['etiqueta'].tolist(),
    test_size=0.2, random_state=42, stratify=df['etiqueta'].tolist())

print('\n== ENTRENANDO MODELO DE GÉNERO LITERARIO ==')
modelo_genero = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS).to(device)

ld_g_tr  = DataLoader(DatasetLiterario(X_g_tr,  y_g_tr,  tokenizer), batch_size=16, shuffle=True)
ld_g_val = DataLoader(DatasetLiterario(X_g_val, y_g_val, tokenizer), batch_size=16)

loop_entrenamiento(modelo_genero, ld_g_tr, ld_g_val, 'modelo_genero.pt', epochs=8)
if os.path.exists('modelo_genero.pt'):
    modelo_genero.load_state_dict(torch.load('modelo_genero.pt', map_location=device))
    print(" Mejor checkpoint de género cargado")
else:
    print(" No se guardó checkpoint de género, usando último epoch")
modelo_genero.eval()

#MODELO TIPO
X_t_tr, X_t_val, y_t_tr, y_t_val = train_test_split(
    textos_tipo, etiquetas_tipo, test_size=0.2, random_state=42, stratify=etiquetas_tipo)

print('\n== ENTRENANDO MODELO DE TIPO DE LECTURA ==')
modelo_tipo = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_TIPOS).to(device)

ld_t_tr  = DataLoader(DatasetLiterario(X_t_tr,  y_t_tr,  tokenizer), batch_size=16, shuffle=True)
ld_t_val = DataLoader(DatasetLiterario(X_t_val, y_t_val, tokenizer), batch_size=16)

loop_entrenamiento(modelo_tipo, ld_t_tr, ld_t_val, 'modelo_tipo.pt', epochs=5)
if os.path.exists('modelo_tipo.pt'):
    modelo_tipo.load_state_dict(torch.load('modelo_tipo.pt', map_location=device))
    print(" Mejor checkpoint de tipo cargado")
else:
    print("  No se guardó checkpoint de tipo, usando último epoch")
modelo_tipo.eval()

print('\n Ambos modelos entrenados y listos')

##  Celda 5 — Sentence-BERT + FAISS para búsqueda semántica

In [ ]:
# SENTENCE-BERT
print('Cargando Sentence-BERT...')
sbert = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print('Generando embeddings del corpus...')
emb_train = sbert.encode(X_g_tr, batch_size=64, show_progress_bar=True).astype('float32')
faiss.normalize_L2(emb_train)

dim    = emb_train.shape[1]
indice = faiss.IndexFlatIP(dim)
indice.add(emb_train)

# ChromaDB
cliente_chroma = chromadb.Client()
coleccion = cliente_chroma.get_or_create_collection('corpus_literario')
for i in range(0, len(X_g_tr), 100):
    coleccion.add(
        ids=[f'doc_{i+j}' for j in range(len(X_g_tr[i:i+100]))],
        embeddings=emb_train[i:i+100].tolist(),
        documents=X_g_tr[i:i+100],
        metadatas=[{'genero': GENEROS[e]} for e in y_g_tr[i:i+100]]
    )

print(f'FAISS: {indice.ntotal} vectores | ChromaDB: {coleccion.count()} documentos')

##  Celda 6 — Motor de Identificación Automática de Autores



In [ ]:
def _limpiar_texto(texto):
    palabras = re.findall(r'\b[a-zA-ZáéíóúÁÉÍÓÚñÑ]{4,}\b', texto)
    stopwords = {'that','this','with','from','they','have','been','were','their',
                 'when','what','which','there','would','could','should','about',
                 'into','over','after','before','more','than','your','very',
                 'para','pero','como','todo','esta','esto','también','porque',
                 'hasta','desde','entre','sobre','tiene','tiene','según','durante'}
    return ' '.join(p for p in palabras if p.lower() not in stopwords)[:80]

#GUTENBERG CATALOG
def buscar_gutenberg(texto):
    query = _limpiar_texto(texto)
    try:
        r = requests.get('https://gutendex.com/books/',
                         params={'search': query, 'languages': 'en,es'}, timeout=10)
        candidatos = []
        for libro in r.json().get('results', [])[:5]:
            autores = libro.get('authors', [])
            if autores:
                n = autores[0].get('name', '')
                if ',' in n:
                    p = n.split(',', 1)
                    n = f'{p[1].strip()} {p[0].strip()}'
                candidatos.append({'autor': n, 'titulo': libro.get('title','N/D'),
                                   'fuente': 'Gutenberg', 'score': libro.get('download_count',0)})
        return candidatos
    except: return []

# OPEN LIBRARY
def buscar_open_library(texto):
    query = _limpiar_texto(texto)
    try:
        r = requests.get('https://openlibrary.org/search.json',
                         params={'q': query, 'limit': 5,
                                 'fields': 'title,author_name,first_publish_year,isbn'},
                         timeout=10)
        candidatos = []
        for doc in r.json().get('docs', [])[:5]:
            autores = doc.get('author_name', [])
            if autores:
                candidatos.append({'autor': autores[0], 'titulo': doc.get('title','N/D'),
                                   'año': doc.get('first_publish_year','N/D'),
                                   'isbn': (doc.get('isbn') or [''])[0],
                                   'fuente': 'OpenLibrary', 'score': 1})
        return candidatos
    except: return []

# GOOGLE BOOKS con ISBN
def buscar_google_books(texto, isbn=None):
    if isbn:
        query = f'isbn:{isbn}'
    else:
        query = f'"{texto[:60]}"'
    try:
        r = requests.get('https://www.googleapis.com/books/v1/volumes',
                         params={'q': query, 'maxResults': 5, 'printType': 'books'},
                         timeout=10)
        candidatos = []
        for item in r.json().get('items', [])[:5]:
            info = item.get('volumeInfo', {})
            autores = info.get('authors', [])
            if autores:
                isbns = [i['identifier'] for i in info.get('industryIdentifiers', [])
                         if i.get('type') in ('ISBN_13','ISBN_10')]
                candidatos.append({'autor': autores[0], 'titulo': info.get('title','N/D'),
                                   'año': info.get('publishedDate','N/D')[:4],
                                   'isbn': isbns[0] if isbns else 'N/D',
                                   'fuente': 'GoogleBooks', 'score': 1})
        return candidatos
    except: return []

# WORLDCAT
def buscar_worldcat(texto):
    query = _limpiar_texto(texto)[:60]
    sru_url = 'http://www.worldcat.org/webservices/catalog/search/sru'
    params = {
        'query': f'srw.kw=\"{query}\"',
        'version': '1.1',
        'operation': 'searchRetrieve',
        'recordSchema': 'info:srw/schema/1/dc-v1.1',
        'maximumRecords': '5',
    }
    try:
        r = requests.get(sru_url, params=params, timeout=12,
                         headers={'User-Agent': 'Letrova/2.0 (academic project)'})
        candidatos = []
        creators = re.findall(r'<[^:]*:creator[^>]*>([^<]+)</[^:]*:creator>', r.text)
        titles   = re.findall(r'<[^:]*:title[^>]*>([^<]+)</[^:]*:title>', r.text)
        for idx, creator in enumerate(creators[:5]):
            candidatos.append({
                'autor':  creator.strip(),
                'titulo': titles[idx].strip() if idx < len(titles) else 'N/D',
                'fuente': 'WorldCat',
                'score':  0.5,
            })
        return candidatos
    except: return []

#WIKIPEDIA PERFIL
def obtener_perfil_wikipedia(nombre_autor):
    nombre_url = nombre_autor.replace(' ', '_')
    for idioma in ['es', 'en']:
        try:
            url = f'https://{idioma}.wikipedia.org/api/rest_v1/page/summary/{nombre_url}'
            r   = requests.get(url, timeout=10)
            d   = r.json()
            if r.status_code == 200 and d.get('extract'):
                resumen = d['extract']
                nac = 'N/D'
                for pais, kws in [
                    ('Colombiano/a',   ['colombian','colombia','colombiano']),
                    ('Estadounidense', ['american','united states']),
                    ('Británico/a',    ['british','english','uk']),
                    ('Ruso/a',         ['russian','russia']),
                    ('Francés/a',      ['french','france']),
                    ('Argentino/a',    ['argentinian','argentina']),
                    ('Chileno/a',      ['chilean','chile']),
                    ('Cubano/a',       ['cuban','cuba']),
                    ('Alemán/a',       ['german','germany']),
                    ('Irlandés/a',     ['irish','ireland']),
                    ('Español/a',      ['spanish','spain','español']),
                    ('Mexicano/a',     ['mexican','mexico']),
                    ('Brasileño/a',    ['brazilian','brazil']),
                    ('Japonés/a',      ['japanese','japan']),
                    ('Italiano/a',     ['italian','italy']),
                    ('Dominicano/a',   ['dominican','dominicana']),
                    ('Venezolano/a',   ['venezuelan','venezuela']),
                    ('Peruano/a',      ['peruvian','peru','peruano']),
                    ('Uruguayo/a',     ['uruguayan','uruguay']),
                ]:
                    if any(k in resumen.lower() for k in kws):
                        nac = pais; break
                epoca = 'N/D'
                for ep, kws in [
                    ('Siglo XIX',   ['19th century','nineteenth','1800','1810','1820','1830','1840','1850','1860','1870','1880','1890']),
                    ('Siglo XX',    ['20th century','twentieth','1900','1910','1920','1930','1940','1950','1960','1970','1980','1990']),
                    ('Siglo XXI',   ['21st century','2000','2010','2020']),
                    ('Siglo XVIII', ['18th century','eighteenth','1700','1710','1720','1730','1740','1750','1760','1770','1780','1790']),
                ]:
                    if any(k in resumen.lower() for k in kws):
                        epoca = ep; break
                return {
                    'descripcion':  d.get('description','N/D'),
                    'resumen':      resumen[:500],
                    'nacionalidad': nac,
                    'epoca':        epoca,
                    'idioma':       idioma,
                    'url':          d.get('content_urls',{}).get('desktop',{}).get('page','N/D')
                }
        except: continue
    return None

#OBRAS DEL AUTOR
def obtener_obras_autor(nombre_autor, max_obras=5):
    try:
        r = requests.get('https://openlibrary.org/search/authors.json',
                         params={'q': nombre_autor}, timeout=10)
        docs = r.json().get('docs', [])
        if not docs: return []
        autor_key = docs[0].get('key','')
        r2 = requests.get(f'https://openlibrary.org/authors/{autor_key}/works.json',
                          params={'limit': max_obras}, timeout=10)
        return [e.get('title','') for e in r2.json().get('entries', []) if e.get('title')][:max_obras]
    except: return []

# FUNCIÓN PRINCIPAL
def identificar_autor_automatico(texto, verbose=True):
    if verbose:
        print('\n' + '─'*55)
        print(' IDENTIFICACIÓN AUTOMÁTICA DE AUTOR')
        print('─'*55)
        print(f'Texto: "{texto[:80]}..."')

    todos_candidatos = []

    # Fuente 1: Gutenberg
    if verbose: print('\n  [1/4] Consultando Gutenberg Catalog...')
    c = buscar_gutenberg(texto); todos_candidatos.extend(c)
    if verbose:
        for x in c[:2]: print(f'        → {x["autor"]} · "{x["titulo"][:45]}"')
        if not c: print('        → Sin resultados')

    # Fuente 2: Open Library
    if verbose: print('  [2/4] Consultando Open Library...')
    c_ol = buscar_open_library(texto); todos_candidatos.extend(c_ol)
    isbn_encontrado = next((x['isbn'] for x in c_ol if x.get('isbn')), None)
    if verbose:
        for x in c_ol[:2]: print(f'        → {x["autor"]} · "{x["titulo"][:45]}" (ISBN: {x.get("isbn","N/D")})')
        if not c_ol: print('        → Sin resultados')

    # Fuente 3: Google Books
    if verbose:
        if isbn_encontrado:
            print(f'  [3/4] Consultando Google Books (ISBN: {isbn_encontrado})...')
        else:
            print('  [3/4] Consultando Google Books (búsqueda por fragmento)...')
    c_gb = buscar_google_books(texto, isbn=isbn_encontrado); todos_candidatos.extend(c_gb)
    if verbose:
        for x in c_gb[:2]: print(f'        → {x["autor"]} · "{x["titulo"][:45]}" (ISBN: {x.get("isbn","N/D")})')
        if not c_gb: print('        → Sin resultados')

    # Fuente 4: WorldCat
    if verbose: print('  [4/4] Consultando WorldCat SRU...')
    c_wc = buscar_worldcat(texto); todos_candidatos.extend(c_wc)
    if verbose:
        for x in c_wc[:2]: print(f'        → {x["autor"]} · "{x["titulo"][:45]}"')
        if not c_wc: print('        → Sin resultados')

    # Consolidar votos
    votos = {}; apariciones = {}
    for c in todos_candidatos:
        autor = ' '.join(w.capitalize() for w in c['autor'].strip().split())
        votos[autor]       = votos.get(autor, 0) + (1 + c.get('score',0) * 0.001)
        apariciones[autor] = apariciones.get(autor, [])
        apariciones[autor].append(c.get('titulo','N/D'))

    if not votos:
        if verbose: print('\n No se encontraron candidatos.')
        return {'autor': 'No identificado', 'confianza': '0%', 'perfil': None}

    ranking     = sorted(votos.items(), key=lambda x: x[1], reverse=True)
    total_votos = sum(v for _, v in ranking)

    if verbose:
        print('\n Candidatos consolidados (4 fuentes):')
        for autor, v in ranking[:4]:
            pct = v / total_votos * 100
            titles = list(set(apariciones[autor]))[:2]
            print(f'     {autor:<32} {pct:.1f}%  — {titles}')

    autor_ganador = ranking[0][0]
    confianza     = ranking[0][1] / total_votos * 100

    if verbose: print(f'\n  Autor identificado: {autor_ganador} ({confianza:.0f}% consenso)')
    perfil_wiki = obtener_perfil_wikipedia(autor_ganador)
    obras       = obtener_obras_autor(autor_ganador)

    resultado = {
        'autor':       autor_ganador,
        'confianza':   f'{round(confianza,1)}%',
        'titulos_ref': list(set(apariciones[autor_ganador]))[:3],
        'obras':       obras,
        'perfil':      perfil_wiki,
        'candidatos':  [(a, round(v/total_votos*100,1)) for a, v in ranking[:3]],
    }

    if verbose:
        print('\n' + '─'*55)
        print(f'  Autor       : {autor_ganador}')
        print(f'  Confianza   : {confianza:.0f}%')
        if perfil_wiki:
            print(f'  Descripción : {perfil_wiki["descripcion"]}')
            print(f'  Nacionalidad: {perfil_wiki["nacionalidad"]}')
            print(f'  Época       : {perfil_wiki["epoca"]}')
            print(f'  Wikipedia   : {perfil_wiki["url"]}')
            print(f'  Resumen     : {perfil_wiki["resumen"][:200]}...')
        if obras:
            print(f'  Obras destac: {", ".join(obras[:3])}')
        print('─'*55)

    return resultado

print('  Motor de identificación automática de autores listo')
print()
print('  Fuentes disponibles (4):')
print('  • Gutendex (catálogo Gutenberg)       — 70.000+ libros')
print('  • Open Library + ISBN                 — 40 millones+ registros')
print('  • Google Books API (ISBN prioritario) — búsqueda precisa')
print('  • WorldCat SRU                        — mayor catálogo mundial')
print('  • Wikipedia                           — perfil biográfico completo')

##  Celda 7 — Prueba del motor de identificación automática

In [ ]:
pruebas = [
    'It was the best of times it was the worst of times it was the age of wisdom',
    'One morning Gregor Samsa woke from troubled dreams to find himself transformed into a monstrous creature',
    'Many years later as he faced the firing squad Colonel Aureliano remembered the afternoon his father showed him ice',
    'To be or not to be that is the question whether tis nobler in the mind to suffer',
    'Call me Ishmael Some years ago never mind how long precisely having little money in my purse',
]

for texto in pruebas:
    resultado = identificar_autor_automatico(texto, verbose=True)
    print()

## Celda 8 — PIPELINE COMPLETO

In [ ]:
#FUNCIONES DE PREDICCIÓN
def predecir_genero(texto):
    enc = tokenizer(texto, max_length=128, padding='max_length',
                    truncation=True, return_tensors='pt')
    with torch.no_grad():
        logits = modelo_genero(
            input_ids=enc['input_ids'].to(device),
            attention_mask=enc['attention_mask'].to(device)
        ).logits
    probs = torch.softmax(logits, dim=1)[0]
    pred  = probs.argmax().item()
    return {'genero': GENEROS[pred], 'confianza': probs[pred].item(),
            'todas': {GENEROS[i]: probs[i].item() for i in range(NUM_LABELS)}}

def predecir_tipo(texto):
    enc = tokenizer(texto, max_length=128, padding='max_length',
                    truncation=True, return_tensors='pt')
    with torch.no_grad():
        logits = modelo_tipo(
            input_ids=enc['input_ids'].to(device),
            attention_mask=enc['attention_mask'].to(device)
        ).logits
    probs = torch.softmax(logits, dim=1)[0]
    pred  = probs.argmax().item()
    return {'tipo': TIPOS_LECTURA[pred], 'confianza': probs[pred].item(),
            'todas': {TIPOS_LECTURA[i]: probs[i].item() for i in range(NUM_TIPOS)}}


# PIPELINE COMPLETO
def clasificar_libro(texto, verbose=True):
    if verbose:
        print('\n' + '═'*65)
        print('   LETROVA — CLASIFICACIÓN INTELIGENTE DE LIBROS')
        print('═'*65)
        print(f'  "{texto[:90]}..."')
        print('═'*65)

    # Género
    rg = predecir_genero(texto)
    if verbose:
        print(f'\n[1]  GÉNERO LITERARIO : {rg["genero"]}  ({rg["confianza"]*100:.1f}%)')
        for g, p in rg['todas'].items():
            bar = '█' * int(p*20)
            mk  = ' ◄' if g == rg['genero'] else ''
            print(f'    {g:<16} {p*100:5.1f}%  {bar}{mk}')

    # Tipo
    rt = predecir_tipo(texto)
    if verbose:
        print(f'\n[2] TIPO DE LECTURA  : {rt["tipo"]}  ({rt["confianza"]*100:.1f}%)')
        for t, p in rt['todas'].items():
            bar = '█' * int(p*20)
            mk  = ' ◄' if t == rt['tipo'] else ''
            print(f'    {t:<18} {p*100:5.1f}%  {bar}{mk}')

    # Autor automático
    if verbose: print()
    ra = identificar_autor_automatico(texto, verbose=verbose)

    if verbose:
        print('\n' + '═'*65)
        print('   ANÁLISIS COMPLETADO')
        print(f'  Género    : {rg["genero"]}')
        print(f'  Tipo      : {rt["tipo"]}')
        print(f'  Autor     : {ra["autor"]}  ({ra["confianza"]} consenso)')
        if ra.get('perfil'):
            print(f'  Nac.      : {(ra.get("perfil") or {}).get("nacionalidad", "N/D")}')
            print(f'  Época     : {(ra.get("perfil") or {}).get("epoca", "N/D")}')
        if ra.get('obras', []):
            print(f'  Obras     : {", ".join(ra["obras"][:3])}')
        print('═'*65)

    return {
        'texto':        texto[:100],
        'genero':       rg['genero'],
        'conf_genero':  f'{rg["confianza"]*100:.1f}%',
        'tipo':         rt['tipo'],
        'conf_tipo':    f'{rt["confianza"]*100:.1f}%',
        'autor':        ra['autor'],
        'conf_autor':   str(ra['confianza']),
        'nacionalidad': (ra.get('perfil') or {}).get('nacionalidad', 'N/D'),
        'epoca':        (ra.get('perfil') or {}).get('epoca', 'N/D'),
        'obras':        ra.get('obras', []),
        'wiki_url':     (ra.get('perfil') or {}).get('url', 'N/D'),
    }

print(' Pipeline completo definido')

## Celda 9 — DEMO: Análisis completo de fragmentos reales

In [ ]:
demos = [
    'It was the best of times it was the worst of times it was the age of wisdom it was the age of foolishness',
    'The little rabbit hopped through the meadow looking for carrots and singing a happy song in the sun',
    'The empirical evidence suggests a significant correlation between early language exposure and literacy outcomes',
    'Call me Ishmael some years ago never mind how long precisely having little money in my purse',
]

resultados = []
for texto in demos:
    r = clasificar_libro(texto, verbose=True)
    resultados.append(r)
    print()

##  Celda 10 — Guardar y descargar modelos

In [ ]:
for nombre in ['modelo_genero.pt', 'modelo_tipo.pt']:
    if os.path.exists(nombre):
        print(f' {nombre} — {os.path.getsize(nombre)/1e6:.1f} MB')
    else:
        print(f' {nombre} — no encontrado')

try:
    from google.colab import files
    files.download('modelo_genero.pt')
    files.download('modelo_tipo.pt')
    print(' Descarga iniciada desde Colab')
except ImportError:
    print(f' Modelos en: {os.getcwd()}')